# Top 150 Python Technical, Verbal & Core Architecture Interview Questions & Answers
### *The Ultimate Master Guide: From CPython Internals to FAANG, Tier-1 Fintech & High-Performance Production Systems*

---

### Structure of This Master Notebook
This notebook is organized into **15 Progressive Conceptual Pillars (150 Questions Total)** sorted by seniority and role expectations:

#### **Junior / Associate Data Scientist / Python Developer (Q1 – Q40)**
1. **Category 1 (Q1 – Q10)**: Python Data Model, Memory, Mutability & CPython Internals
2. **Category 2 (Q11 – Q20)**: Core Data Structures (Lists, Dicts, Sets, Deques, Heaps) & Complexity
3. **Category 3 (Q21 – Q30)**: Control Flow, Comprehensions, Walrus Operator & Slicing Mechanics
4. **Category 4 (Q31 – Q40)**: Functions, Scope (LEGB), Closures & Arguments

#### **Mid-Level / Senior Data Scientist / Backend & ML Engineer (Q41 – Q100)**
5. **Category 5 (Q41 – Q50)**: Iterators, Generators & Memory-Efficient Streaming
6. **Category 6 (Q51 – Q60)**: Object-Oriented Programming, MRO & Dunder Methods
7. **Category 7 (Q61 – Q70)**: Decorators & Metaprogramming Patterns
8. **Category 8 (Q71 – Q80)**: Context Managers & Resource Management
9. **Category 9 (Q81 – Q90)**: Error Handling, Custom Exceptions & Defensive Programming
10. **Category 10 (Q91 – Q100)**: Standard Library Superpowers (bisect, secrets, mmap, enum, zoneinfo)

#### **Staff / Lead / Tier-1 Quant / High-Performance Systems Architect (Q101 – Q150)**
11. **Category 11 (Q101 – Q110)**: Python Concurrency: Threading, Multiprocessing & GIL Mechanics
12. **Category 12 (Q111 – Q120)**: Asynchronous Programming (asyncio, TaskGroup, Event Loop)
13. **Category 13 (Q121 – Q130)**: Metaprogramming, Descriptors, Metaclasses & __init_subclass__
14. **Category 14 (Q131 – Q140)**: CPython Internals, GC & Low-Level Memory Layout
15. **Category 15 (Q141 – Q150)**: High-Performance, Compilation (Numba, Cython, CFFI) & Production Anti-Patterns

---
Each question includes:
- **Verbal Explanation**: Crisp, interview-ready conceptual summary (plain whiteboard text, no LaTeX).
- **Tricky Follow-up Question & Answer**: The exact follow-up or trap interviewers test you on.
- **Runnable Python Code Block**: Practical, executable demonstration with explicit labeled print statements for all created concepts.



## Category 1: Python Data Model, Memory, Mutability & CPython Internals


### Q1: How does Python manage memory and what is the difference between mutable and immutable objects?

#### Verbal Answer:

Python treats everything as an object managed on the private heap:

1. **Object Structure (PyObject)**:
   * Every Python object contains at least two header fields:
     * `ob_refcnt`: Reference count for memory deallocation.
     * `ob_type`: Pointer to the object's type descriptor.

2. **Immutable Objects (`int`, `float`, `str`, `tuple`, `frozenset`, `bytes`)**:
   * Value cannot be changed in-place after instantiation.
   * Any modification creates a brand new object in memory with a new `id()`.
   * Thread-safe and hashable (usable as dictionary keys).

3. **Mutable Objects (`list`, `dict`, `set`, `bytearray`)**:
   * Value can be mutated in-place without altering the object's memory address (`id()`).
   * Unhashable (cannot be dictionary keys or set elements).

---

#### Tricky Follow-up:

* **Question:** *Can an immutable tuple contain a mutable list, and what happens if you try to mutate that list inside the tuple?*

* **Answer:** Yes. The tuple only holds fixed memory references (pointers); the nested list can be mutated in-place (`t[0].append(1)`), though `t` itself remains unhashable.



In [ ]:
# 1. Immutable integer vs Mutable list
a = 100
id_a_before = id(a)
a += 1
id_a_after = id(a)

lst = [1, 2, 3]
id_lst_before = id(lst)
lst.append(4)
id_lst_after = id(lst)

# 2. Tuple containing a mutable list
mixed_tuple = ([10, 20], 'static')
mixed_tuple[0].append(30)

print("1. Immutable int ID changed after += 1:", id_a_before != id_a_after)
print("2. Mutable list ID unchanged after append:", id_lst_before == id_lst_after)
print("3. Tuple with mutated nested list:", mixed_tuple)


### Q2: What is the fundamental difference between the 'is' identity operator and the '==' equality operator?

#### Verbal Answer:

Identity vs Equality comparison in Python:

1. **`==` (Value Equality)**:
   * Calls the `__eq__()` dunder method.
   * Evaluates whether two objects have equivalent **contents/values**, regardless of their physical memory address.

2. **`is` (Identity Comparison)**:
   * Compares memory addresses directly: `id(a) == id(b)`.
   * Evaluates whether two variable names point to the **exact same physical object in RAM**.
   * Fastest comparison (O(1) pointer comparison in CPython).

---

#### Tricky Follow-up:

* **Question:** *Why is `is` strictly required when checking for `None` (`if x is None:`) instead of `==`?*

* **Answer:** `None` is a CPython singleton, and custom classes can override `__eq__()` to return `True` when compared to `None`, producing false positives with `==`.



In [ ]:
# Value equality vs Identity
list1 = [1, 2, 3]
list2 = [1, 2, 3]
list3 = list1

print("1. list1 == list2 (Equal values):", list1 == list2)
print("2. list1 is list2 (Different objects in RAM):", list1 is list2)
print("3. list1 is list3 (Same object reference):", list1 is list3)
print("4. Memory address list1:", id(list1))
print("5. Memory address list2:", id(list2))


### Q3: What is integer interning and string interning in CPython, and why does 'a = 256; b = 256; a is b' return True while 257 may not?

#### Verbal Answer:

Interning is an internal CPython optimization to save memory and accelerate lookups:

1. **Small Integer Caching**:
   * CPython pre-allocates an array of singleton integer objects for the range **`[-5, 256]`** at startup.
   * Any variable assigned an integer in this range points to the shared singleton in memory (`is` evaluates to `True`).
   * Numbers outside `[-5, 256]` (e.g. 257) allocate distinct memory objects in interactive sessions.

2. **String Interning**:
   * Python automatically interns strings that look like Python identifiers (ASCII letters, digits, underscores) and dictionary keys for fast pointer-equality lookups in dictionaries.

---

#### Tricky Follow-up:

* **Question:** *How can you manually force Python to intern a dynamic runtime string?*

* **Answer:** Use `sys.intern(my_string)` to store it in CPython's internal interning dictionary.



In [ ]:
import sys

# Small integer caching [-5, 256]
x = 256
y = 256
print("1. 256 is cached singleton (x is y):", x is y)

# Dynamic string interning
s1 = sys.intern('custom_dynamic_identifier_string')
s2 = sys.intern('custom_dynamic_identifier_string')
print("2. Manually interned strings share identity (s1 is s2):", s1 is s2)


### Q4: How do Reference Counting and Cyclic Garbage Collection (Generational GC) work together in CPython?

#### Verbal Answer:

CPython uses a two-tier memory management system:

1. **Primary: Reference Counting (Deterministic)**:
   * Every object tracks active references (`ob_refcnt`).
   * Increment on assignment, decrement on `del` or scope exit.
   * When `refcnt == 0`, memory is **immediately deallocated**.

2. **Secondary: Generational Cyclic Garbage Collector**:
   * Resolves **Reference Cycles** (e.g. Object A points to Object B, and Object B points to Object A; both have `refcnt=1` even when orphaned).
   * Organizes objects into 3 generations (Gen 0: young, Gen 1: middle, Gen 2: long-lived).
   * Scans pointer graphs to detect isolated unreachable islands of objects and frees them.

---

#### Tricky Follow-up:

* **Question:** *What happens to reference cycle collection if an object defines a legacy `__del__()` destructor in Python versions prior to 3.4?*

* **Answer:** In Python <3.4 (PEP 442), reference cycles with `__del__` were uncollectable and placed in `gc.garbage`. Modern Python 3.4+ collects them safely.



In [ ]:
import sys
import gc

# 1. Reference counting demonstration
obj = []
print("1. Initial refcount (passed to getrefcount):", sys.getrefcount(obj))
alias = obj
print("2. Refcount after alias:", sys.getrefcount(obj))

# 2. Cyclic reference creation
class Node:
    def __init__(self):
        self.cycle = None

n1 = Node()
n2 = Node()
n1.cycle = n2
n2.cycle = n1

del n1, n2  # Objects orphaned but cyclically linked!
unreachable = gc.collect()
print("3. Cyclic unreachable objects collected by GC:", unreachable)


### Q5: What is shallow copy vs deep copy, and how does copy.deepcopy() handle recursive references?

#### Verbal Answer:

Duplication mechanics via the `copy` module:

1. **Shallow Copy (`copy.copy(x)` or `list.copy()` or `x[:]`)**:
   * Creates a new outer container, but inserts references to the **same child objects** found in the original.
   * Mutating a nested list in the shallow copy mutates the original!

2. **Deep Copy (`copy.deepcopy(x)`)**:
   * Recursively duplicates the outer container AND all nested child objects down the entire hierarchy.

3. **Recursive Reference Safety**:
   * `deepcopy` maintains an internal `memo` dictionary `{id(obj): cloned_obj}` to prevent infinite loops when copying cyclic data structures.

---

#### Tricky Follow-up:

* **Question:** *Does `copy.copy()` or `copy.deepcopy()` duplicate immutable objects like strings or integers?*

* **Answer:** No. Immutable objects are reused directly without duplication, returning the object itself to conserve memory.



In [ ]:
import copy

original = [[1, 2, 3], 'static_text']
shallow = copy.copy(original)
deep = copy.deepcopy(original)

# Mutate nested list
original[0].append(999)

print("1. Original after nested mutation:", original)
print("2. Shallow Copy (Reflects mutation!):", shallow)
print("3. Deep Copy (Independent, unaffected):", deep)


### Q6: How are variables passed to functions in Python (Pass-by-Value vs Pass-by-Reference vs Pass-by-Assignment)?

#### Verbal Answer:

Python uses **Pass-by-Object-Reference** (also known as **Pass-by-Assignment**):

1. **Mechanism**:
   * Function arguments are passed as **bindings of names to shared object references**.

2. **Mutating In-Place (Mutable Arguments)**:
   * Calling mutator methods (e.g. `arg.append(x)` or `arg['key'] = val`) modifies the caller's object in-place because both names point to the same memory block.

3. **Rebinding Arguments (`arg = new_val`)**:
   * Reassigning the argument name inside the function merely binds the local name to a new object; it has zero effect on the caller's original variable.

---

#### Tricky Follow-up:

* **Question:** *What happens when you use `a += [4]` versus `a = a + [4]` on a list passed into a function?*

* **Answer:** `a += [4]` invokes `__iadd__()` which mutates the list in-place; `a = a + [4]` creates a new list and rebinds the local variable name.



In [ ]:
def modify_data(lst, num):
    lst.append(100)   # In-place mutation (affects caller)
    num = num + 50    # Local rebinding (does NOT affect caller)

my_list = [1, 2]
my_num = 10

modify_data(my_list, my_num)

print("1. List after function call (Mutated):", my_list)
print("2. Number after function call (Unchanged):", my_num)


### Q7: Why are mutable default arguments (def func(x=[])) dangerous and what is the standard pythonic idiom?

#### Verbal Answer:

The Dangerous Mutable Default Parameter Trap:

1. **The Root Cause**:
   * Default parameter expressions are evaluated **once at function definition time**, NOT each time the function is called.
   * If the default value is mutable (`list`, `dict`), all subsequent function invocations that use the default share the **exact same object reference** in memory.

2. **The Pythonic Idiom**:
   * Set default to `None` and instantiate dynamically inside the function body:
   ```python
   def append_item(item, target_list=None):
       if target_list is None:
           target_list = []
       target_list.append(item)
       return target_list
   ```

---

#### Tricky Follow-up:

* **Question:** *Can mutable default arguments be intentionally leveraged for anything useful?*

* **Answer:** Yes. Prior to `functools.lru_cache`, developers used default dictionaries `def memoized(x, cache={}):` as private function-level memoization caches.



In [ ]:
# 1. Buggy pattern with mutable default
def buggy_append(val, container=[]):
    container.append(val)
    return container

# 2. Pythonic safe pattern
def safe_append(val, container=None):
    if container is None:
        container = []
    container.append(val)
    return container

print("1. Buggy Call 1:", buggy_append('A'))
print("2. Buggy Call 2 (Accumulated state!):", buggy_append('B'))
print("3. Safe Call 1: ", safe_append('A'))
print("4. Safe Call 2: ", safe_append('B'))


### Q8: What is the Global Interpreter Lock (GIL) and why does it prevent multi-threaded CPU speedups in CPython?

#### Verbal Answer:

The Global Interpreter Lock (GIL) is a mutex synchronization lock in CPython:

1. **Why It Exists**:
   * CPython's memory management (reference counting) is not thread-safe. The GIL prevents race conditions by allowing only **one native CPU thread** to execute Python bytecode at any given moment.

2. **Impact on Workloads**:
   * **CPU-Bound Tasks (e.g. loops, math)**: Multi-threading provides **zero speedup** (often runs slower due to thread context switching overhead). Must use `multiprocessing` or C/Cython extensions.
   * **I/O-Bound Tasks (e.g. network requests, disk reads)**: Multi-threading works excellently because threads release the GIL while waiting on socket/file I/O.

3. **Modern Python 3.13 Free-Threaded Mode (PEP 703)**:
   * Introduces experimental build options allowing disabling the GIL using mimalloc and biased reference counting.

---

#### Tricky Follow-up:

* **Question:** *Do C-extensions like NumPy release the GIL during execution?*

* **Answer:** Yes. NumPy, SciPy, and PyTorch release the GIL during heavy numerical computations, enabling true multi-core parallel execution via OpenMP/BLAS.



In [ ]:
import sys

print("1. Python Version:", sys.version.split()[0])
print("2. GIL Rule: I/O-bound uses Threading / Asyncio; CPU-bound uses Multiprocessing / Polars / Numba.")


### Q9: How does Python compile source code into Bytecode (.pyc) and how does the CPython Virtual Machine execute it?

#### Verbal Answer:

The Python Execution Pipeline:

1. **Compilation Phase**:
   * Source Code (`.py`) is parsed into an Abstract Syntax Tree (AST).
   * AST is compiled into **Python Bytecode** (stored in `__pycache__/*.pyc` files for caching).
   * Bytecode consists of compact 2-byte instructions (opcode + argument).

2. **Execution Phase (CPython Virtual Machine)**:
   * The CPython VM is a **Stack-Based Virtual Machine**.
   * Evaluates bytecode instructions by pushing and popping objects onto the execution evaluation stack (`dis` module allows inspecting opcodes).

---

#### Tricky Follow-up:

* **Question:** *How can you inspect the exact bytecode instructions generated for a Python function?*

* **Answer:** Use the built-in `dis.dis(function_name)` module to disassemble function bytecode.



In [ ]:
import dis

def calculate_tax(price, rate=0.08):
    total = price * (1 + rate)
    return total

print("Disassembled CPython Bytecode:")
dis.dis(calculate_tax)


### Q10: What are __slots__ in Python classes and how do they reduce memory overhead by 60%?

#### Verbal Answer:

`__slots__` overrides default dynamic instance attribute storage:

1. **Default Class Memory Model**:
   * Every standard Python instance stores its attributes in an internal dictionary (`instance.__dict__`).
   * Dictionaries allocate hash table buckets, consuming substantial RAM (~150+ bytes per instance overhead).

2. **`__slots__` Optimization**:
   * Replaces `__dict__` with a fixed-size C-style array of attribute pointers.
   * Decreases memory usage by up to **60-70%** and slightly accelerates attribute access.
   * Prevents dynamic assignment of undeclared attributes (raises `AttributeError`).

---

#### Tricky Follow-up:

* **Question:** *What happens if a child class inherits from a parent class with `__slots__` but does not define `__slots__` itself?*

* **Answer:** The child class instance will create an instance `__dict__`, losing the memory savings for child attributes.



In [ ]:
import sys

class StandardPoint:
    def __init__(self, x, y):
        self.x = x
        self.y = y

class SlottedPoint:
    __slots__ = ('x', 'y')
    def __init__(self, x, y):
        self.x = x
        self.y = y

p_std = StandardPoint(10, 20)
p_slot = SlottedPoint(10, 20)

print("1. Standard instance has __dict__:", hasattr(p_std, '__dict__'))
print("2. Slotted instance has NO __dict__:", not hasattr(p_slot, '__dict__'))
print("3. Standard instance size:", sys.getsizeof(p_std) + sys.getsizeof(p_std.__dict__), "bytes")
print("4. Slotted instance size: ", sys.getsizeof(p_slot), "bytes")


## Category 2: Core Data Structures & Complexity


### Q11: How are Python Lists implemented in CPython (Dynamic Array) and what are the time complexities of list operations?

#### Verbal Answer:

Python lists are implemented as **Variable-Length Contiguous Arrays of Pointers** (`PyListObject`):

1. **Over-Allocation Strategy**:
   * When growing, CPython over-allocates memory geometrically (`0, 4, 8, 16, 24, 32, 40, 52, ...`) to avoid reallocating memory on every insert.

2. **Time Complexities**:
   * **`append()` & `pop()` from end**: **O(1) amortized** time.
   * **Indexing / Slicing lookup (`lst[i]`)**: **O(1)** pointer offset arithmetic.
   * **`insert(0, val)` & `pop(0)` (from front)**: **O(N)** because all subsequent elements must be shifted in memory.
   * **`val in lst` (Linear Search)**: **O(N)**.
   * **`sort()` (Timsort)**: **O(N log N)** time, O(N) space.

---

#### Tricky Follow-up:

* **Question:** *Which data structure should be used instead of a list when you need O(1) insertions and pops from the beginning?*

* **Answer:** Use `collections.deque` (Doubly Linked List of memory blocks) which provides O(1) `appendleft()` and `popleft()`.



In [ ]:
import time
from collections import deque

n = 50000
py_list = []
py_deque = deque()

# Benchmark inserting at front: list.insert(0) vs deque.appendleft()
t0 = time.perf_counter()
for i in range(n):
    py_list.insert(0, i)
t_list = time.perf_counter() - t0

t0 = time.perf_counter()
for i in range(n):
    py_deque.appendleft(i)
t_deque = time.perf_counter() - t0

print(f"1. List insert(0) O(N^2) time:   {t_list*1000:.2f} ms")
print(f"2. Deque appendleft() O(N) time: {t_deque*1000:.2f} ms")
print(f"Deque is {t_list / t_deque:.1f}x faster for front operations!")


### Q12: How is the Python Dictionary implemented internally and why are dicts insertion-ordered since Python 3.7?

#### Verbal Answer:

Python Dictionaries are implemented via **Compact Hash Tables** (Raymond Hettinger design):

1. **The Compact Hash Table Architecture**:
   * **`indices` Array**: A sparse array of hash table buckets storing small integer indices.
   * **`entries` Array**: A dense, contiguous array of `[hash, key, value]` structs in order of insertion.

2. **Key Benefits**:
   * **30-40% Memory Reduction**: Eliminates wasted empty dictionary bucket slots.
   * **Guaranteed Insertion Order**: Iterating over `dict.keys()` or `dict.values()` preserves insertion sequence natively.
   * **O(1) Average Lookup**: Fast hash table lookup with Open Addressing and Quadratic Perturbation collision resolution.

---

#### Tricky Follow-up:

* **Question:** *What happens during a hash collision in a Python dictionary?*

* **Answer:** CPython uses open addressing with a perturbation formula: `perturb >>= 5; j = (5*j + 1 + perturb) & mask` to probe next candidate slots.



In [ ]:
# Insertion order preservation in modern Python dictionaries
user_dict = {}
user_dict['user_id'] = 101
user_dict['role'] = 'Admin'
user_dict['status'] = 'Active'

print("Dictionary keys in exact insertion order:", list(user_dict.keys()))
print("Dictionary values in exact insertion order:", list(user_dict.values()))


### Q13: What requirements must an object fulfill to be used as a Dictionary Key or Set Element (Hashable Protocol)?

#### Verbal Answer:

The Hashable Protocol requires an object to implement:

1. **`__hash__()`**: Must return an integer that remains **constant across the object's entire lifetime**.
2. **`__eq__()`**: Must compare equality with other objects. If `a == b`, then `hash(a) == hash(b)` MUST hold true.

Rules:
* All built-in immutable types (`int`, `str`, `tuple` containing hashables) are hashable.
* Mutable objects (`list`, `dict`, `set`) are unhashable (`TypeError: unhashable type`).
* Custom class instances are hashable by default (hashing their `id()` memory address).

---

#### Tricky Follow-up:

* **Question:** *If you override `__eq__()` in a custom class without defining `__hash__()`, what does Python do?*

* **Answer:** Python sets `__hash__ = None`, making instances of the class unhashable to prevent breaking hash table invariants.



In [ ]:
class StockAsset:
    def __init__(self, ticker, exchange):
        self.ticker = ticker
        self.exchange = exchange

    def __eq__(self, other):
        return isinstance(other, StockAsset) and (self.ticker, self.exchange) == (other.ticker, other.exchange)

    def __hash__(self):
        return hash((self.ticker, self.exchange))

asset1 = StockAsset('AAPL', 'NASDAQ')
asset2 = StockAsset('AAPL', 'NASDAQ')

portfolio_set = {asset1, asset2}
print("1. Asset1 hash equals Asset2 hash:", hash(asset1) == hash(asset2))
print("2. Set deduplication succeeded (Length == 1):", len(portfolio_set) == 1)


### Q14: How do Sets operate in Python and what are the mathematical set operations (&, |, -, ^)?

#### Verbal Answer:

Sets are implemented as hash tables containing only keys (no values):

1. **Characteristics**:
   * Stores unique, unordered, hashable elements.
   * O(1) average membership testing (`x in my_set`), insertions, and deletions.

2. **Mathematical Operators**:
   * **Intersection (`a & b`)**: Elements in both `a` AND `b`.
   * **Union (`a | b`)**: Elements in `a` OR `b`.
   * **Difference (`a - b`)**: Elements in `a` but NOT in `b`.
   * **Symmetric Difference (`a ^ b`)**: Elements in `a` or `b`, but NOT both.

---

#### Tricky Follow-up:

* **Question:** *What is the difference between `set.union(other_iterable)` and `set | other_set`?*

* **Answer:** The method `.union()` accepts any iterable (lists, tuples, generators), while the bitwise operator `|` strictly requires both operands to be `set` objects.



In [ ]:
set_a = {1, 2, 3, 4}
set_b = {3, 4, 5, 6}

print("1. Union (a | b):               ", set_a | set_b)
print("2. Intersection (a & b):        ", set_a & set_b)
print("3. Difference (a - b):          ", set_a - set_b)
print("4. Symmetric Difference (a ^ b):", set_a ^ set_b)


### Q15: What is a frozenset and when is it required over a standard set?

#### Verbal Answer:

`frozenset` is the immutable counterpart to `set`:

1. **Immutability & Hashability**:
   * Cannot be modified in-place (no `add()`, `remove()`, or `discard()`).
   * Because it is immutable, it is **hashable**!

2. **Use Cases**:
   * Storing sets as **keys in a dictionary**.
   * Creating **nested sets** (a set containing other sets).

---

#### Tricky Follow-up:

* **Question:** *Can you use a `frozenset` as a dictionary key to represent multi-item graph edges or combinations?*

* **Answer:** Yes. `frozenset(['NodeA', 'NodeB'])` is the standard pattern for representing undirected graph edges in adjacency dictionaries.



In [ ]:
# Nested sets and dictionary keys using frozenset
edge1 = frozenset(['NYC', 'LON'])
edge2 = frozenset(['LON', 'NYC'])  # Identical undirected edge!

flight_distances = {
    edge1: 5570
}

print("1. Undirected edge lookup matches regardless of order:", flight_distances[edge2])
print("2. Set of frozensets (Nested Sets):", {frozenset([1, 2]), frozenset([3, 4])})


### Q16: What is collections.defaultdict and how does it prevent KeyError exceptions?

#### Verbal Answer:

`collections.defaultdict(default_factory)` automates missing key initialization:

1. **Mechanism**:
   * When accessing a non-existent key `d[key]`, `defaultdict` automatically invokes `default_factory()` without raising a `KeyError`, inserts the returned value, and returns it.

2. **Common Factories**:
   * `defaultdict(list)`: Grouping elements into lists without checking `if key not in d`.
   * `defaultdict(int)`: Counting occurrences (defaults to 0).
   * `defaultdict(set)`: Deduplicating relationships.

---

#### Tricky Follow-up:

* **Question:** *Does accessing a missing key using `.get(key)` on a `defaultdict` trigger the `default_factory`?*

* **Answer:** No. `.get(key)` bypasses `__missing__()` and returns `None` without inserting the key into the dictionary.



In [ ]:
from collections import defaultdict

# Group transactions by customer
orders = [('CustA', 100), ('CustB', 200), ('CustA', 150)]

grouped_orders = defaultdict(list)
for cust, amount in orders:
    grouped_orders[cust].append(amount)

print("Grouped orders via defaultdict(list):", dict(grouped_orders))


### Q17: How does collections.Counter work for frequency distributions and most common elements?

#### Verbal Answer:

`collections.Counter` is a specialized dictionary subclass for counting hashable items:

1. **Core Capabilities**:
   * Automatically counts frequencies of iterables: `Counter(['a', 'b', 'a'])`.
   * Returns `0` for missing keys instead of raising `KeyError`.

2. **Key Methods**:
   * `c.most_common(k)`: Returns the top `k` most frequent elements in O(N log k) heap time.
   * `c1 + c2` & `c1 - c2`: Vectorized multiset arithmetic.

---

#### Tricky Follow-up:

* **Question:** *What is the time complexity of `Counter.most_common(k)`?*

* **Answer:** It uses `heapq.nlargest()` internally, running in O(N log k) time where N is the number of distinct elements.



In [ ]:
from collections import Counter

words = ['apple', 'banana', 'apple', 'orange', 'banana', 'apple', 'kiwi']
freq = Counter(words)

print("1. Counter distribution:", freq)
print("2. Top 2 most common:    ", freq.most_common(2))
print("3. Missing key count:    ", freq['unseen_fruit'])


### Q18: What is collections.OrderedDict and when is it still preferred over standard Python dict?

#### Verbal Answer:

`collections.OrderedDict` vs standard Python dictionaries:

1. **Historical Context**:
   * Standard dicts became insertion-ordered in Python 3.7+.

2. **Why `OrderedDict` is Still Specialized**:
   * **Order-Sensitive Equality**: `OrderedDict([('a', 1), ('b', 2)]) != OrderedDict([('b', 2), ('a', 1)])` (standard dicts evaluate as `True`).
   * **`move_to_end(key, last=True/False)`**: Fast O(1) rearrangement of keys (essential for building LRU Caches).
   * **`popitem(last=False)`**: Pops the first (FIFO) element in O(1) time.

---

#### Tricky Follow-up:

* **Question:** *How can you implement a simple Least Recently Used (LRU) Cache using `OrderedDict`?*

* **Answer:** On access, call `od.move_to_end(key)`; on insert exceeding capacity, call `od.popitem(last=False)`.



In [ ]:
from collections import OrderedDict

# LRU Cache pattern using OrderedDict
class LRUCache:
    def __init__(self, capacity):
        self.cache = OrderedDict()
        self.capacity = capacity

    def get(self, key):
        if key not in self.cache:
            return -1
        self.cache.move_to_end(key)
        return self.cache[key]

    def put(self, key, value):
        if key in self.cache:
            self.cache.move_to_end(key)
        self.cache[key] = value
        if len(self.cache) > self.capacity:
            self.cache.popitem(last=False)

lru = LRUCache(2)
lru.put('A', 1)
lru.put('B', 2)
lru.get('A')    # A is accessed (B becomes least recently used)
lru.put('C', 3) # Evicts B

print("LRU Cache Keys after operations:", list(lru.cache.keys()))


### Q19: What is collections.namedtuple and how does it compare to standard tuples, dictionaries, and dataclasses?

#### Verbal Answer:

`namedtuple` provides lightweight, memory-efficient record objects:

1. **Characteristics**:
   * Subclass of standard `tuple`.
   * Access fields by name (`point.x`) or by index (`point[0]`).
   * **Zero Memory Overhead**: Consumes identical memory to a standard tuple (no `__dict__`).
   * Fully immutable and hashable.

2. **Comparison**:
   * vs `dict`: Consumes ~70% less memory; immutable.
   * vs `dataclass`: `namedtuple` is strictly a tuple (untyped unless using `typing.NamedTuple`); `dataclass` supports mutability and type inheritance.

---

#### Tricky Follow-up:

* **Question:** *How do you convert a `namedtuple` to a standard Python dictionary?*

* **Answer:** Call `point._asdict()`.



In [ ]:
from collections import namedtuple

Transaction = namedtuple('Transaction', ['tx_id', 'amount', 'currency'])

tx = Transaction(tx_id=9876, amount=150.75, currency='USD')

print("1. Named attribute access:", tx.amount)
print("2. Positional tuple index:", tx[1])
print("3. Exported as dictionary:", tx._asdict())


### Q20: What is the heapq module in Python (Min-Heap) and how do you implement a Max-Heap?

#### Verbal Answer:

`heapq` implements a **Binary Min-Heap** using a standard Python list:

1. **Min-Heap Properties**:
   * `heap[0]` is always the **smallest element** in O(1) time.
   * `heapq.heappush(heap, item)`: O(log N) insertion.
   * `heapq.heappop(heap)`: O(log N) extraction of the minimum.
   * `heapq.heapify(lst)`: Converts an unsorted list to a valid heap in **linear O(N) time**.

2. **Max-Heap Implementation**:
   * Invert numerical values by multiplying by `-1` upon insertion, and invert back upon popping (`-heapq.heappop(max_heap)`).

---

#### Tricky Follow-up:

* **Question:** *Why is `heapq.heapify(lst)` O(N) time complexity instead of O(N log N)?*

* **Answer:** Because it uses Floyd's bottom-up sift-down algorithm, which does fewer swaps on lower levels containing the majority of nodes.



In [ ]:
import heapq

# Min-Heap demonstration
data = [50, 10, 30, 20, 40]
heapq.heapify(data)

smallest = heapq.heappop(data)
heapq.heappush(data, 15)

# Max-Heap using inverted numbers
max_heap = [-x for x in [10, 50, 30]]
heapq.heapify(max_heap)
largest = -heapq.heappop(max_heap)

print("1. Smallest extracted from Min-Heap:", smallest)
print("2. Heap state after push(15):", data)
print("3. Largest extracted from Max-Heap:", largest)


## Category 3: Control Flow, Comprehensions & Slicing Mechanics


### Q21: How do List, Dict, and Set Comprehensions execute and why are they faster than standard for-loops?

#### Verbal Answer:

Comprehensions are optimized syntactic constructs for constructing containers:

1. **Why Comprehensions are Faster than For-Loops**:
   * Executed at C-level in CPython via specialized bytecode (`LIST_APPEND`, `MAP_ADD`, `SET_ADD`).
   * Bypasses repeated Python-level method lookup and function call overhead (`list.append`).

2. **Syntax**:
   * List: `[x**2 for x in items if x > 0]`
   * Set: `{x**2 for x in items}`
   * Dict: `{k: v for k, v in pairs}`

---

#### Tricky Follow-up:

* **Question:** *What is the danger of nesting 3 or more `for` loops inside a single comprehension?*

* **Answer:** Severe loss of code readability and high risk of creating accidental O(N^3) Cartesian products.



In [ ]:
# List, Dict, and Set Comprehensions
nums = [1, 2, 3, 4, 5, 2, 3]

sq_list = [x**2 for x in nums if x % 2 == 0]
unique_sets = {x for x in nums}
val_map = {f"k_{x}": x*10 for x in set(nums)}

print("1. Filtered List Comprehension:", sq_list)
print("2. Deduplicated Set Comprehension:", unique_sets)
print("3. Key-Value Dict Comprehension:", val_map)


### Q22: What is the Walrus Operator (:=) introduced in Python 3.8 and what are its primary production use cases?

#### Verbal Answer:

The **Assignment Expression Operator (`:=`)** assigns values to variables inside expressions:

1. **Syntax**: `name := expr`

2. **Core Use Cases**:
   * **While Loops (Streaming/Regex/I/O)**:
     ```python
     while (chunk := file.read(1024)):
         process(chunk)
     ```
   * **Comprehension Filtering without Duplicate Computation**:
     `[y for x in data if (y := heavy_calc(x)) > threshold]`
   * **Conditional Branching**:
     `if (n := len(items)) > 100: print(f"Large batch: {n}")`

---

#### Tricky Follow-up:

* **Question:** *Does the walrus operator create a new scope inside list comprehensions?*

* **Answer:** No. Variables assigned with the walrus operator inside a comprehension leak into the surrounding enclosing function scope.



In [ ]:
data_samples = ['AAPL', 'MSFT', 'GOOGLE_CORP', 'AMZN']

# Walrus operator inside list comprehension to avoid calling len() twice
long_tickers = [name for s in data_samples if (name := s.strip().upper()) and len(name) > 4]

print("Long Tickers identified via Walrus Operator (:=):", long_tickers)


### Q23: How does Extended Slicing (slice(start, stop, step)) operate on sequences and how do you reverse sequences efficiently?

#### Verbal Answer:

Sequence slicing uses `seq[start:stop:step]` or `slice()` objects:

1. **Semantics**:
   * `start`: Inclusive start index.
   * `stop`: Exclusive stop index.
   * `step`: Index stride (negative step reverses direction).

2. **Reversing Sequences**:
   * `seq[::-1]`: Creates a reversed shallow copy in O(N) time.
   * `reversed(seq)`: Returns a memory-efficient **reverse iterator** in O(1) time without allocating a new sequence.

3. **Slice Objects**:
   * Reusable: `FIRST_THREE = slice(0, 3); data[FIRST_THREE]`.

---

#### Tricky Follow-up:

* **Question:** *What happens if slice indices are out of range (e.g. `lst[0:1000]` on a 3-element list)?*

* **Answer:** Python slices handle out-of-bounds indices gracefully without raising `IndexError`, clipping to the sequence boundaries.



In [ ]:
items = [10, 20, 30, 40, 50, 60]

# Extended slicing and reversal
even_pos = items[::2]
reversed_copy = items[::-1]
out_of_bounds = items[0:1000]

print("1. Every 2nd element (items[::2]):", even_pos)
print("2. Reversed copy (items[::-1]):    ", reversed_copy)
print("3. Out-of-bounds slice:            ", out_of_bounds)


### Q24: What is Extended Iterable Unpacking (* and ** operators) and how does it unpack sequences and dictionaries?

#### Verbal Answer:

Starred unpacking allows flexible positional and keyword decomposition:

1. **Positional Sequence Unpacking (`*`)**:
   * `first, *middle, last = [1, 2, 3, 4, 5]`
   * Merging lists: `combined = [*list1, *list2]`

2. **Keyword Dictionary Unpacking (`**`)**:
   * Merging dictionaries (PEP 448): `merged = {**dict1, **dict2}` (values in `dict2` overwrite `dict1`).
   * In Python 3.9+, the dictionary union operator `dict1 | dict2` is also supported.

---

#### Tricky Follow-up:

* **Question:** *Can you have two starred expressions `*a, *b = items` on the left-hand side of an assignment?*

* **Answer:** No. Python raises a `SyntaxError: two starred expressions in assignment` due to ambiguous length partitioning.



In [ ]:
# Starred unpacking
data = [100, 200, 300, 400, 500]
first, *middle, last = data

# Dictionary unpacking
defaults = {'theme': 'dark', 'notifications': True}
user_prefs = {'theme': 'light'}
final_config = {**defaults, **user_prefs}

print("1. First:", first, "| Middle:", middle, "| Last:", last)
print("2. Merged Config:", final_config)


### Q25: How does the else clause behave when paired with for loops, while loops, and try-except blocks?

#### Verbal Answer:

The `else` clause has specialized semantics beyond `if-else`:

1. **`for ... else` / `while ... else`**:
   * Executes **ONLY if the loop completes normally without encountering a `break` statement**.
   * Standard pythonic idiom for search loops (eliminates boolean `found = False` flags).

2. **`try ... except ... else`**:
   * Executes **ONLY if NO exception was raised** inside the `try` block.
   * Best practice for code that should run only on success, keeping the `try` block minimal.

---

#### Tricky Follow-up:

* **Question:** *Does `for ... else` execute if the input iterable is completely empty `[]`?*

* **Answer:** Yes. Since the loop completed without encountering a `break`, the `else` block executes immediately.



In [ ]:
# 1. for-else search pattern
target = 42
for num in [10, 20, 30]:
    if num == target:
        print("Found target!")
        break
else:
    print("1. Target 42 not found in loop (for-else executed cleanly).")

# 2. try-except-else pattern
try:
    val = int("100")
except ValueError:
    print("Conversion failed!")
else:
    print("2. Conversion succeeded without exception (try-else executed):", val)


### Q26: What is Structural Pattern Matching (match-case) introduced in Python 3.10?

#### Verbal Answer:

Structural Pattern Matching (`match-case`, PEP 634) provides type, shape, and value matching:

1. **Beyond Switch-Case**:
   * Matches nested patterns, sequence lengths, class types, and extracts bound variables simultaneously.

2. **Guard Clauses (`if`)**:
   * `case [x, y] if x > y: ...`

3. **Wildcard (`_`)**:
   * Default catch-all pattern matching anything.

---

#### Tricky Follow-up:

* **Question:** *How does `case Point(x=0, y=y):` match custom class attributes?*

* **Answer:** It checks `isinstance(obj, Point)` and extracts `obj.y` when `obj.x == 0`.



In [ ]:
def route_command(command):
    match command:
        case ['QUIT']:
            return "Terminating session."
        case ['MOVE', x, y] if int(x) >= 0:
            return f"Moving to positive coordinate ({x}, {y})"
        case ['TRANSFER', int(amount), str(currency)]:
            return f"Transferring {amount} {currency}"
        case _:
            return "Unknown command pattern."

print("1. Match command:", route_command(['MOVE', 10, 20]))
print("2. Match transfer:", route_command(['TRANSFER', 500, 'USD']))


### Q27: How do zip(), enumerate(), and reversed() optimize iteration pipelines?

#### Verbal Answer:

Built-in iterator utilities:

1. **`enumerate(iterable, start=0)`**:
   * Yields pairs `(index, value)` without manual counter variables.

2. **`zip(*iterables, strict=False)`**:
   * Combines parallel iterables into tuples `(a1, b1), (a2, b2)`.
   * Python 3.10+ `strict=True` raises `ValueError` if iterables have unequal lengths.

3. **`reversed(seq)`**:
   * Returns a reverse iterator over sequences in O(1) space.

---

#### Tricky Follow-up:

* **Question:** *What happens in standard `zip(a, b)` without `strict=True` if `a` has 10 elements and `b` has 3?*

* **Answer:** It silently truncates at 3 elements. To fill missing values instead, use `itertools.zip_longest()`.



In [ ]:
names = ['Alice', 'Bob', 'Charlie']
scores = [95, 88, 92]

# 1. enumerate with custom start index
enumerated = list(enumerate(names, start=1))

# 2. strict zip
paired = list(zip(names, scores, strict=True))

print("1. Enumerate starting at 1:", enumerated)
print("2. Strict Zip Paired Data: ", paired)


### Q28: How do any() and all() implement Short-Circuit Evaluation across boolean iterables?

#### Verbal Answer:

Short-circuit evaluation in logical aggregators:

1. **`any(iterable)`**:
   * Returns `True` as soon as the **first truthy element** is encountered, terminating evaluation immediately without inspecting remaining items.
   * Returns `False` on empty iterables.

2. **`all(iterable)`**:
   * Returns `False` as soon as the **first falsy element** is encountered.
   * Returns `True` on empty iterables (**Vacuous Truth**).

---

#### Tricky Follow-up:

* **Question:** *Why does `all([])` evaluate to `True`?*

* **Answer:** In formal logic, a statement about all elements of an empty set is vacuously true because no element exists that can falsify the condition.



In [ ]:
# Short-circuit verification
def expensive_check(n):
    return n > 10

numbers = [2, 4, 15, 8, 100]

# any() stops immediately at 15
has_large = any(expensive_check(x) for x in numbers)
all_positive = all(x > 0 for x in numbers)
empty_all = all([])

print("1. any() found value > 10: ", has_large)
print("2. all() verified positive:  ", all_positive)
print("3. all([]) on empty iterable:", empty_all)


### Q29: What is the difference between sorted() and list.sort() in return values and in-place mutation?

#### Verbal Answer:

Sorting mechanics in Python:

1. **`list.sort(key=..., reverse=...)`**:
   * Modifies the list **in-place**.
   * Returns `None` (preventing accidental reassignment bugs).
   * Restricted strictly to `list` objects.

2. **`sorted(iterable, key=..., reverse=...)`**:
   * Accepts **any iterable** (tuples, dicts, generators).
   * Returns a **brand new sorted list**, leaving the original input completely untouched.

---

#### Tricky Follow-up:

* **Question:** *What sorting algorithm powers `sorted()` and `list.sort()` in Python?*

* **Answer:** Timsort (Powersort in Python 3.11+), an adaptive, stable hybrid of Merge Sort and Insertion Sort with O(N log N) worst-case and O(N) best-case time.



In [ ]:
original_list = [5, 2, 8, 1, 9]

# sorted() creates new list
sorted_new = sorted(original_list)

# list.sort() mutates in-place
original_list.sort(reverse=True)

print("1. New Sorted List via sorted():", sorted_new)
print("2. Original List Mutated in-place:", original_list)


### Q30: How do you flatten deeply nested lists in Python using itertools.chain vs recursive generators?

#### Verbal Answer:

Techniques for flattening nested structures:

1. **1-Level Shallow Flattening**:
   * `itertools.chain.from_iterable(nested_list)` or `[x for sub in nested for x in sub]`.
   * Fastest O(N) C-level generator.

2. **Arbitrarily Deep Nested Flattening**:
   * Recursive generator using `yield from`:
   ```python
   def deep_flatten(items):
       for item in items:
           if isinstance(item, (list, tuple)):
               yield from deep_flatten(item)
           else:
               yield item
   ```

---

#### Tricky Follow-up:

* **Question:** *Why must you check `isinstance(item, (list, tuple))` rather than `isinstance(item, Iterable)` in recursive flattening?*

* **Answer:** Because strings are iterables of characters (`'abc'` -> `'a'`, `'b'`, `'c'`). Checking generic `Iterable` causes infinite recursion on single characters (`'a'[0] == 'a'`).



In [ ]:
import itertools

# 1. 1-Level flattening with itertools.chain
level1 = [[1, 2], [3, 4], [5]]
flat1 = list(itertools.chain.from_iterable(level1))

# 2. Arbitrary deep flattening
def deep_flatten(items):
    for item in items:
        if isinstance(item, (list, tuple)):
            yield from deep_flatten(item)
        else:
            yield item

deep_nested = [1, [2, [3, [4, 5]], 6], 7]
flat_deep = list(deep_flatten(deep_nested))

print("1. 1-Level Flattened via itertools.chain:", flat1)
print("2. Deep Recursive Flattened:            ", flat_deep)


## Category 4: Functions, Scope (LEGB), Closures & Arguments


### Q31: What is the LEGB Rule for variable scope resolution in Python (Local, Enclosing, Global, Built-in)?

#### Verbal Answer:

The LEGB rule defines Python's variable lookup order:

1. **L - Local**: Names assigned within the currently executing function.
2. **E - Enclosing**: Names defined in outer/parent enclosing functions (nested scopes/closures).
3. **G - Global**: Names defined at the module top level or declared via `global`.
4. **B - Built-in**: Names preloaded into Python's built-in namespace (`len`, `range`, `ValueError`).

If a variable name is not found across any of these four scopes, Python raises a `NameError`.

---

#### Tricky Follow-up:

* **Question:** *Can you accidentally shadow a built-in function by defining a local variable with the same name (e.g. `list = [1, 2]`)?*

* **Answer:** Yes. Because 'Local' precedes 'Built-in', calling `list()` afterwards raises a `TypeError: 'list' object is not callable`.



In [ ]:
global_val = "G"

def outer():
    enclosing_val = "E"
    def inner():
        local_val = "L"
        return f"{local_val} -> {enclosing_val} -> {global_val} -> len={len([1])}"
    return inner()

print("LEGB Scope Resolution Chain:", outer())


### Q32: What is the difference between the global and nonlocal keywords?

#### Verbal Answer:

Explicit scope rebinding declarations:

1. **`global var_name`**:
   * Declares that assignments to `var_name` bind directly to the **module top-level global scope**.

2. **`nonlocal var_name`**:
   * Declares that assignments to `var_name` bind to the **nearest enclosing function scope** (excluding the global namespace).
   * Essential for stateful closures and custom factory functions.

---

#### Tricky Follow-up:

* **Question:** *What happens if you use `nonlocal` on a variable that exists only in the global scope?*

* **Answer:** Python raises a `SyntaxError: no binding for nonlocal 'x' found` because `nonlocal` strictly requires an enclosing function scope.



In [ ]:
def counter_factory(start=0):
    count = start
    def increment():
        nonlocal count  # Rebinds to enclosing function scope
        count += 1
        return count
    return increment

c = counter_factory(10)
print("1. Counter Call 1 (nonlocal state):", c())
print("2. Counter Call 2 (nonlocal state):", c())


### Q33: What is a Closure in Python and what conditions are required for a closure to form?

#### Verbal Answer:

A **Closure** is a nested function that retains access to variables in its lexical enclosing scope even after the outer function has finished executing:

Conditions:
1. Must have a nested function.
2. The nested function must reference a variable defined in the enclosing function.
3. The enclosing function must return the nested function object.

Python stores closed-over variables inside the function's **`__closure__`** attribute as `cell` objects.

---

#### Tricky Follow-up:

* **Question:** *How can you inspect the closed-over cell objects of a closure function in Python?*

* **Answer:** Inspect `func.__closure__[i].cell_contents`.



In [ ]:
def make_multiplier(factor):
    def multiply(x):
        return x * factor  # Closes over 'factor'
    return multiply

double = make_multiplier(2)
triple = make_multiplier(3)

print("1. double(5):", double(5))
print("2. triple(5):", triple(5))
print("3. Closed-over cell content:", double.__closure__[0].cell_contents)


### Q34: What is the Late Binding Closure Gotcha and how do you resolve it (lambda in loops)?

#### Verbal Answer:

The classic Late Binding Closure Trap:

1. **The Problem**:
   * Python functions look up variables in their enclosing scope **at call time**, NOT at definition time.
   * `funcs = [lambda: i for i in range(3)]` -> All 3 lambdas look up `i` when called; since the loop finished with `i = 2`, calling any lambda returns `2`.

2. **The Solution**:
   * Bind the loop variable as a **default argument** `lambda i=i: i`, which evaluates and freezes `i` at definition time.

---

#### Tricky Follow-up:

* **Question:** *Can `functools.partial` also be used to fix late binding?*

* **Answer:** Yes. `functools.partial(func, i)` binds the current value of `i` eagerly.



In [ ]:
# 1. Buggy late binding closures
buggy_funcs = [lambda: i for i in range(3)]

# 2. Fixed eager binding via default arguments
fixed_funcs = [lambda i=i: i for i in range(3)]

print("1. Buggy results (all return final i=2):", [f() for f in buggy_funcs])
print("2. Fixed results (eagerly bound 0, 1, 2): ", [f() for f in fixed_funcs])


### Q35: How do *args, **kwargs, Keyword-Only Arguments (*), and Positional-Only Arguments (/) work?

#### Verbal Answer:

Python's Comprehensive Argument Syntax (PEP 570):

`def func(pos_only, /, standard, *, kw_only, **kwargs):`

1. **`/` (Positional-Only)**: Arguments before `/` **must** be passed positionally (cannot use `name=val`).
2. **`*args`**: Captures arbitrary extra positional arguments as a `tuple`.
3. **`*` (Keyword-Only Boundary)**: Arguments after `*` **must** be passed as keyword arguments (`arg=val`).
4. **`**kwargs`**: Captures arbitrary extra keyword arguments as a `dict`.

---

#### Tricky Follow-up:

* **Question:** *Why does the Python standard library use positional-only arguments `/` in C-accelerated functions like `abs(x)` or `len(obj)`?*

* **Answer:** To allow renaming parameter names in C extensions without breaking callers and to prevent argument keyword dictionary overhead.



In [ ]:
def configure_system(host, port, /, timeout=30, *, secure=True, **extra):
    return {
        'host': host,
        'port': port,
        'timeout': timeout,
        'secure': secure,
        'extra': extra
    }

config = configure_system("127.0.0.1", 8080, 60, secure=True, debug=True, log_level="INFO")
print("Structured Function Arguments Parsing Result:", config)


### Q36: What are First-Class Functions and Higher-Order Functions in Python?

#### Verbal Answer:

Functions as First-Class Citizens in Python:

1. **First-Class Properties**:
   * Functions can be passed as arguments to other functions.
   * Functions can be returned as values from other functions.
   * Functions can be assigned to variables and stored in data structures (lists, dicts).

2. **Higher-Order Functions**:
   * Any function that takes one or more functions as arguments or returns a function (e.g. `map()`, `filter()`, `sorted(key=...)`, decorators).

---

#### Tricky Follow-up:

* **Question:** *What is the return type of `map()` and `filter()` in Python 3?*

* **Answer:** They return lazy **iterators** rather than materialized lists to conserve memory.



In [ ]:
# First-class function dispatch table
def add(a, b): return a + b
def multiply(a, b): return a * b

math_ops = {'+': add, '*': multiply}

res1 = math_ops['+'](10, 5)
res2 = math_ops['*'](10, 5)

print("1. Addition via dispatch table:      ", res1)
print("2. Multiplication via dispatch table:", res2)


### Q37: How do Lambda Functions differ from def functions in syntax, limitations, and bytecode?

#### Verbal Answer:

`lambda` vs `def` functions:

1. **`lambda` Characteristics**:
   * Anonymous single-expression functions: `lambda x, y: x + y`.
   * Cannot contain statements (`if/else` ternary is allowed, but not `if` statements, `while`, `try-except`, or assignments).
   * Implicitly returns the evaluated expression.

2. **Bytecode Equality**:
   * Under the hood, CPython compiles both `lambda` and `def` into identical `PyCodeObject` structures; the only difference is `lambda.__name__ == '<lambda>'`.

---

#### Tricky Follow-up:

* **Question:** *Why does PEP 8 advise against assigning lambdas to variable names (`f = lambda x: x + 1`)?*

* **Answer:** PEP 8 recommends `def f(x): return x + 1` because named `def` functions provide descriptive tracebacks and docstring support.



In [ ]:
# Sorting complex structures via lambda
assets = [('NVDA', 120), ('AAPL', 180), ('MSFT', 420)]
sorted_by_price = sorted(assets, key=lambda pair: pair[1])

print("Assets Sorted by Price via Lambda:", sorted_by_price)


### Q38: What is Currying and Partial Application (functools.partial) and when is it used?

#### Verbal Answer:

Fixing function arguments for modular pipelines:

1. **Partial Application (`functools.partial(func, *args, **kwargs)`)**:
   * Freezes a portion of a function's arguments, producing a new callable with fewer required parameters.

2. **Use Cases**:
   * Adapting multi-argument functions for single-argument APIs like `map()` or event callbacks.
   * Parameterizing database connection factories or ML hyperparameters.

---

#### Tricky Follow-up:

* **Question:** *What is the difference between Currying and Partial Application?*

* **Answer:** Currying transforms a function of N arguments into a chain of N single-argument functions `f(a)(b)(c)`; Partial application fixes some arguments returning a function of lower arity.



In [ ]:
from functools import partial

def power(base, exponent):
    return base ** exponent

# Partial application creating specialized functions
square = partial(power, exponent=2)
cube = partial(power, exponent=3)

print("1. square(5):", square(5))
print("2. cube(5):  ", cube(5))


### Q39: How do Type Hints and the typing module (Union, Optional, Any, Callable, TypeVar) enhance Python codebases?

#### Verbal Answer:

Type Hints (PEP 484) enable static analysis (via mypy / pyright) without runtime performance penalties:

1. **Key Types**:
   * `Union[int, str]` / `int | str` (Python 3.10+): Multiple valid types.
   * `Optional[T]` / `T | None`: Value can be type T or None.
   * `Callable[[int, int], str]`: Function accepting two ints and returning a string.
   * `TypeVar('T')`: Generic types for preserving relationships between inputs and outputs.

2. **Zero Runtime Overhead**:
   * Type annotations are stored in `func.__annotations__` metadata and ignored by the CPython VM during execution.

---

#### Tricky Follow-up:

* **Question:** *How can you inspect type annotations at runtime taking postponed annotations (PEP 563 `from __future__ import annotations`) into account?*

* **Answer:** Use `inspect.get_annotations(obj)` (Python 3.10+) or `typing.get_type_hints(obj)`.



In [ ]:
from typing import TypeVar, List

T = TypeVar('T')

def get_first_element(container: List[T]) -> T | None:
    return container[0] if container else None

print("1. First element int:   ", get_first_element([10, 20, 30]))
print("2. First element string:", get_first_element(['Alpha', 'Beta']))
print("3. Function annotations metadata:", get_first_element.__annotations__)


### Q40: What is recursion depth limit in Python (sys.setrecursionlimit) and why does CPython not support Tail Call Optimization (TCO)?

#### Verbal Answer:

Python Recursion Mechanics:

1. **Recursion Limit**:
   * CPython enforces a recursion depth limit (default 1000 frames) via `sys.getrecursionlimit()` to prevent C stack overflow segfaults (`RecursionError`).

2. **Why Python Rejects Tail Call Optimization (TCO)**:
   * Guido van Rossum intentionally omitted TCO to **preserve complete stack tracebacks for debugging and auditing**.
   * Python emphasizes iterative constructs (`for`, `while`, generators) over deep recursion.

---

#### Tricky Follow-up:

* **Question:** *How do you convert a deep recursive algorithm into an iterative algorithm to avoid RecursionError?*

* **Answer:** Use an explicit heap-allocated Python `list` as a stack simulator (`stack = [initial_state]`).



In [ ]:
import sys

print("1. Default CPython Recursion Limit:", sys.getrecursionlimit())

# Safe iterative Fibonacci vs recursive
def fib_iterative(n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a

print("2. Iterative Fibonacci(100):", fib_iterative(100))


## Category 5: Iterators, Generators & Memory-Efficient Streaming


### Q41: What is the difference between an Iterable and an Iterator in Python (Iterable vs Iterator Protocol)?

#### Verbal Answer:

The Iterator Protocol Architecture:

1. **Iterable**:
   * An object capable of returning an iterator.
   * Implements `__iter__()` (or legacy `__getitem__()`).
   * Examples: `list`, `tuple`, `dict`, `str`.

2. **Iterator**:
   * A stateful stream object representing a sequence of values.
   * Implements `__next__()` to produce the next value, raising `StopIteration` when exhausted.
   * Implements `__iter__()` returning `self` (an iterator is itself an iterable).

---

#### Tricky Follow-up:

* **Question:** *What happens if you pass an iterator to a `for` loop twice?*

* **Answer:** The second `for` loop produces 0 elements because iterators are stateful and can only be consumed once.



In [ ]:
lst = [10, 20, 30]

# Iterable produces a new Iterator
it = iter(lst)

print("1. next(it):", next(it))
print("2. next(it):", next(it))
print("3. next(it):", next(it))

# Iterator exhausted
try:
    next(it)
except StopIteration:
    print("4. Caught StopIteration: Iterator exhausted cleanly.")


### Q42: How do Generator Functions (yield) and Generator Expressions differ from regular functions?

#### Verbal Answer:

Generator mechanics and lazy evaluation:

1. **`yield` Keyword**:
   * Suspends function execution, returns intermediate value to caller, and preserves local stack frame state for resumption.

2. **Memory Efficiency**:
   * Generates values **on-demand (lazy evaluation)** in O(1) space, rather than allocating full collections in RAM.

3. **Generator Expressions**:
   * Compact syntax: `(x**2 for x in huge_range)`.

---

#### Tricky Follow-up:

* **Question:** *How can you check if an object is a generator function versus a generator object?*

* **Answer:** Use `inspect.isgeneratorfunction(func)` and `inspect.isgenerator(gen_obj)`.



In [ ]:
import sys

# List comprehension (materializes 1M ints in RAM)
# Generator expression (O(1) memory pointer stream)
gen_stream = (x**2 for x in range(1000000))

print("1. Generator memory footprint:", sys.getsizeof(gen_stream), "bytes")
print("2. First 3 elements extracted on-demand:")
print("   Val 1:", next(gen_stream))
print("   Val 2:", next(gen_stream))
print("   Val 3:", next(gen_stream))


### Q43: How does yield from simplify generator delegation and sub-generator communication?

#### Verbal Answer:

`yield from` (PEP 380) establishes a direct bidirectional channel to a subgenerator:

1. **Subgenerator Delegation**:
   * Equivalent to `for item in subgen: yield item`, but written concisely as `yield from subgen`.

2. **Bidirectional Value Routing**:
   * Routes `.send()`, `.throw()`, and `.close()` directly to the inner subgenerator.
   * Automatically captures the **return value** of the subgenerator: `result = yield from subgen()`.

---

#### Tricky Follow-up:

* **Question:** *What happens when a subgenerator executes `return 'Done'` inside a `yield from` block?*

* **Answer:** The string `'Done'` is assigned directly as the expression value of `val = yield from subgenerator()`.



In [ ]:
def sub_sequence():
    yield 'A'
    yield 'B'
    return 42

def master_pipeline():
    yield 'Start'
    return_val = yield from sub_sequence()
    yield f'Subgenerator returned: {return_val}'
    yield 'End'

print("Pipeline Stream via yield from:")
for item in master_pipeline():
    print("  ->", item)


### Q44: How do generator methods .send(), .throw(), and .close() turn generators into Coroutines?

#### Verbal Answer:

Coroutines via enhanced generator methods:

1. **`.send(value)`**:
   * Injects data into a paused generator, becoming the value of the active `val = yield` expression.

2. **`.throw(ExceptionType)`**:
   * Raises an exception inside the generator at the current yield point.

3. **`.close()`**:
   * Raises `GeneratorExit` inside the generator to trigger cleanup blocks (`finally: ...`).

---

#### Tricky Follow-up:

* **Question:** *What must you do before calling `gen.send(value)` for the first time?*

* **Answer:** You must 'prime' the generator by calling `next(gen)` or `gen.send(None)` to advance execution to the first `yield`.



In [ ]:
def running_averager():
    total = 0.0
    count = 0
    average = 0.0
    while True:
        val = yield average
        total += val
        count += 1
        average = total / count

avg_coroutine = running_averager()
next(avg_coroutine)  # Prime the coroutine

print("1. Send 10 -> Avg:", avg_coroutine.send(10))
print("2. Send 20 -> Avg:", avg_coroutine.send(20))
print("3. Send 30 -> Avg:", avg_coroutine.send(30))
avg_coroutine.close()


### Q45: How do you build custom iterables with __iter__() and __next__() dunder methods?

#### Verbal Answer:

Implementing the custom Iterator Protocol:

1. **Class Architecture**:
   * `__iter__()`: Returns the iterator object (usually `self` or a fresh iterator instance).
   * `__next__()`: Returns the next element or raises `StopIteration` when sequence ends.

2. **Reusable Iterable Pattern**:
   * To allow multiple independent iterations, have `__iter__()` return a brand new iterator instance rather than `self`.

---

#### Tricky Follow-up:

* **Question:** *Why should a custom collection class separate the Collection from the Iterator into two distinct classes?*

* **Answer:** To allow multiple concurrent, independent iterators over the same collection without their cursor state colliding.



In [ ]:
class Countdown:
    def __init__(self, start):
        self.current = start

    def __iter__(self):
        return self

    def __next__(self):
        if self.current <= 0:
            raise StopIteration
        val = self.current
        self.current -= 1
        return val

print("Custom Iterator Countdown:", list(Countdown(4)))


### Q46: What are the core tools in itertools (count, cycle, repeat, islice, chain, groupby, accumulate)?

#### Verbal Answer:

`itertools` provides C-accelerated composable iterator primitives:

1. **Infinite Streams**:
   * `count(start, step)`: Infinite arithmetic progression.
   * `cycle(iterable)`: Infinite repetition of sequence.
   * `repeat(elem, n)`: Yields element `n` times.

2. **Slicing & Chaining**:
   * `islice(it, start, stop, step)`: Slices any generator without loading prior elements into RAM.
   * `chain(*iterables)`: Sequentially chains multiple iterables.

3. **Transformations**:
   * `accumulate(it, func=add)`: Running cumulative reductions.
   * `groupby(it, key=...)`: Groups consecutive keys (data must be sorted first!).

---

#### Tricky Follow-up:

* **Question:** *Why does `itertools.groupby()` fail to group duplicate keys that are not adjacent?*

* **Answer:** `groupby()` generates a new group whenever the key changes; it does not maintain a global hash table. Input MUST be pre-sorted by the grouping key.



In [ ]:
import itertools

# 1. islice on infinite count
counter = itertools.count(start=10, step=5)
first_3 = list(itertools.islice(counter, 3))

# 2. accumulate running product
nums = [1, 2, 3, 4]
cum_prod = list(itertools.accumulate(nums, lambda a, b: a * b))

# 3. groupby on sorted data
data = [('A', 1), ('A', 2), ('B', 3)]
grouped = {k: [val for _, val in g] for k, g in itertools.groupby(data, key=lambda x: x[0])}

print("1. islice on count():", first_3)
print("2. accumulate product:", cum_prod)
print("3. itertools.groupby: ", grouped)


### Q47: How do itertools.product, permutations, combinations, and combinations_with_replacement work?

#### Verbal Answer:

Combinatorial Iterators in `itertools`:

1. **`product(*iterables, repeat=1)`**:
   * Computes Cartesian product (equivalent to nested `for` loops).

2. **`permutations(iterable, r=None)`**:
   * All possible ordered arrangements of length `r` where order matters (`(A, B) != (B, A)`).

3. **`combinations(iterable, r)`**:
   * All unordered subsets of length `r` without replacement (`(A, B) == (B, A)`).

4. **`combinations_with_replacement(iterable, r)`**:
   * Subsets where individual elements can be repeated.

---

#### Tricky Follow-up:

* **Question:** *How many combinations are generated by `combinations(range(10), 3)`?*

* **Answer:** 10! / (3! * 7!) = 120 combinations.



In [ ]:
import itertools

chars = ['A', 'B', 'C']

prod = list(itertools.product(['X', 'Y'], [1, 2]))
perms = list(itertools.permutations(chars, 2))
combs = list(itertools.combinations(chars, 2))

print("1. Cartesian Product (2x2):", prod)
print("2. Permutations (Order matters):", perms)
print("3. Combinations (Order ignored):", combs)


### Q48: How do you implement a streaming processing pipeline for processing 50GB log files in constant O(1) RAM?

#### Verbal Answer:

Composing generator pipelines for massive data streams:

1. **Pipeline Architecture**:
   * Layer 1 (Source Generator): Reads file line-by-line (`(line for line in file)`).
   * Layer 2 (Filter Generator): Discards irrelevant lines.
   * Layer 3 (Parsing Generator): Extracts JSON/regex capture groups.
   * Layer 4 (Reduction Sink): Aggregates final metrics.

2. **Memory Complexity**:
   * Exactly **O(1) memory** because only one line passes through the pipeline at any instant.

---

#### Tricky Follow-up:

* **Question:** *Why is `file.readlines()` strictly forbidden in big data streaming pipelines?*

* **Answer:** `readlines()` loads every line of the entire file into a giant Python list in RAM simultaneously, causing immediate `MemoryError` crashes.



In [ ]:
# Simulated streaming log pipeline
raw_logs = (f"2023-10-02 INFO Response 200 latency={i}ms" if i % 2 == 0 
            else f"2023-10-02 ERROR Failed 500 latency={i*10}ms" for i in range(100))

# 1. Filter errors
errors = (line for line in raw_logs if "ERROR" in line)

# 2. Extract latency
latencies = (int(line.split("latency=")[1].replace("ms", "")) for line in errors)

# 3. Aggregate
total_error_latency = sum(latencies)

print("Constant RAM Streaming Pipeline Computed Total Latency:", total_error_latency)


### Q49: What is the iter(callable, sentinel) two-argument form of iter()?

#### Verbal Answer:

Two-Argument Sentinel Iterator `iter(callable, sentinel)`:

1. **Mechanism**:
   * Repeatedly calls `callable()` with zero arguments on each step.
   * Yields the returned value until the return value equals `sentinel`, at which point it raises `StopIteration`.

2. **Primary Use Case**:
   * Reading fixed-size blocks from files or sockets until an empty byte marker `b''` is hit:
   ```python
   for block in iter(lambda: f.read(4096), b''):
       process(block)
   ```

---

#### Tricky Follow-up:

* **Question:** *Can you use two-argument `iter()` with a random number generator to stream numbers until a target appears?*

* **Answer:** Yes: `iter(lambda: random.randint(1, 10), 7)` streams random integers until 7 is generated.



In [ ]:
# Stream reader using sentinel iterator
data_stream = iter(['chunk1', 'chunk2', 'STOP', 'chunk3'].pop, 'STOP')

collected = []
for chunk in iter(lambda: next(data_stream), 'STOP'):
    collected.append(chunk)

print("Chunks collected before hitting STOP sentinel:", collected)


### Q50: How does contextlib.contextmanager convert generators into Context Managers?

#### Verbal Answer:

Decorator-based Context Manager construction:

1. **Structure**:
   * Wrap a generator containing a single `yield`.
   * Code **before** `yield` executes as `__enter__()`.
   * The yielded value is bound to the `as variable` target.
   * Code **after** `yield` (inside `finally: ...`) executes as `__exit__()`.

---

#### Tricky Follow-up:

* **Question:** *What happens if an exception is raised inside the `with` block of a generator context manager?*

* **Answer:** The exception is re-raised inside the generator at the `yield` statement. You MUST wrap the `yield` in `try ... finally` to guarantee cleanup.



In [ ]:
from contextlib import contextmanager
import time

@contextmanager
def execution_timer(label):
    start = time.perf_counter()
    try:
        yield
    finally:
        elapsed = time.perf_counter() - start
        print(f"[{label}] Elapsed Time: {elapsed*1000:.3f} ms")

with execution_timer("Sample Matrix Math"):
    res = sum(x**2 for x in range(50000))

print("Execution Result:", res)


## Category 6: Object-Oriented Programming, MRO & Dunder Methods


### Q51: How does Method Resolution Order (MRO) and the C3 Linearization Algorithm resolve multiple inheritance?

#### Verbal Answer:

Method Resolution Order (MRO) defines method lookup hierarchy in multiple inheritance:

1. **C3 Linearization Algorithm**:
   * Guarantees 3 properties:
     1. Children precede their parents.
     2. Precedence of parents matches declaration order in class header `class Child(ParentA, ParentB):`.
     3. Monotonicity (order of superclasses is preserved across the entire inheritance graph).

2. **Inspecting MRO**:
   * `Class.mro()` or `Class.__mro__` returns the exact tuple order of classes evaluated.

---

#### Tricky Follow-up:

* **Question:** *What causes Python to reject a class definition with `TypeError: Cannot create a consistent method resolution order (MRO)`?*

* **Answer:** When inheritance relationships create cyclic or contradictory precedence orders that violate C3 linearization.



In [ ]:
class A: pass
class B(A): pass
class C(A): pass
class D(B, C): pass

print("Method Resolution Order (MRO) for Class D:")
for idx, cls in enumerate(D.mro(), 1):
    print(f"  {idx}. {cls.__name__}")


### Q52: How does super() work dynamically with MRO and cooperative multiple inheritance?

#### Verbal Answer:

`super()` is dynamic MRO navigation, NOT a static link to the direct parent:

1. **Dynamic Dispatch**:
   * `super().method()` delegates execution to the **next class in the MRO chain** of the active instance, which may be a sibling class rather than a direct parent!

2. **Cooperative Multiple Inheritance**:
   * In a Diamond inheritance hierarchy, `super()` guarantees every class method in the diamond executes **exactly once** in linear sequence.

---

#### Tricky Follow-up:

* **Question:** *Why should every method in a cooperative multiple inheritance hierarchy forward `*args, **kwargs` to `super()`?*

* **Answer:** Because a class cannot predict which sibling class will follow it in the runtime MRO, so it must forward all arguments cooperatively.



In [ ]:
class Root:
    def execute(self):
        return ["Root"]

class Left(Root):
    def execute(self):
        return ["Left"] + super().execute()

class Right(Root):
    def execute(self):
        return ["Right"] + super().execute()

class Diamond(Left, Right):
    def execute(self):
        return ["Diamond"] + super().execute()

d = Diamond()
print("Cooperative Multiple Inheritance Execution Chain:", d.execute())


### Q53: What is the difference between @staticmethod, @classmethod, and regular instance methods?

#### Verbal Answer:

Method types in Python classes:

1. **Instance Method**:
   * Receives `self` (the instance object).
   * Reads and modifies instance and class state.

2. **`@classmethod`**:
   * Receives `cls` (the class object itself).
   * Reads and modifies class state across all instances.
   * Standard for **Alternative Constructors** (e.g. `User.from_json()`, `Date.from_timestamp()`).

3. **`@staticmethod`**:
   * Receives neither `self` nor `cls`.
   * Plain isolated utility function bound to the class namespace.

---

#### Tricky Follow-up:

* **Question:** *Why is `@classmethod` preferred over `@staticmethod` when creating factory constructors for inherited classes?*

* **Answer:** `@classmethod` receives the actual subclass `cls` at runtime, ensuring factory methods return instances of the subclass rather than the hardcoded base class.



In [ ]:
class Order:
    base_currency = "USD"

    def __init__(self, order_id, amount):
        self.order_id = order_id
        self.amount = amount

    @classmethod
    def from_csv_string(cls, record):
        order_id, amount = record.split(",")
        return cls(int(order_id), float(amount))

    @staticmethod
    def is_valid_currency(curr):
        return curr in ["USD", "EUR", "GBP"]

order = Order.from_csv_string("101,450.50")

print("1. Instantiated via @classmethod constructor:", order.order_id, order.amount)
print("2. Evaluated via @staticmethod:              ", Order.is_valid_currency("USD"))


### Q54: What is the difference between __str__() and __repr__() in the Python Data Model?

#### Verbal Answer:

Object String Representations:

1. **`__repr__()` (Developer Representation)**:
   * Goal: **Unambiguous**, explicit description for debugging and logging.
   * Best practice: Should look like valid Python code to recreate the object (`Point(x=10, y=20)`).
   * Fallback: If `__str__` is not defined, Python falls back to `__repr__`.

2. **`__str__()` (User Representation)**:
   * Goal: **Readable**, user-friendly presentation for end users (`print(obj)`).

---

#### Tricky Follow-up:

* **Question:** *What does `print([my_obj])` output when printing an object inside a list?*

* **Answer:** Python containers always call `__repr__()` on their contained items, even when the container itself is printed with `print()`.



In [ ]:
class Portfolio:
    def __init__(self, ticker, shares):
        self.ticker = ticker
        self.shares = shares

    def __str__(self):
        return f"{self.shares} shares of {self.ticker}"

    def __repr__(self):
        return f"Portfolio(ticker={self.ticker!r}, shares={self.shares!r})"

p = Portfolio('AAPL', 50)

print("1. print(p) calls __str__:  ", str(p))
print("2. repr(p) calls __repr__:  ", repr(p))
print("3. Contained in list calls __repr__:", [p])


### Q55: How do @property, getter, setter, and deleter implement Encapsulation and Data Validation?

#### Verbal Answer:

Pythonic Encapsulation via Properties:

1. **`@property` Decorator**:
   * Exposes methods with attribute-access syntax (`obj.price` instead of `obj.get_price()`).

2. **`@price.setter`**:
   * Intercepts assignment (`obj.price = 100`) to execute validation logic, type checks, or range constraints.

3. **`@price.deleter`**:
   * Intercepts `del obj.price` for resource cleanup.

---

#### Tricky Follow-up:

* **Question:** *Can a property be made strictly read-only?*

* **Answer:** Yes. Define only the `@property` getter without defining a `@setter`. Attempting to assign raises `AttributeError: can't set attribute`.



In [ ]:
class BankAccount:
    def __init__(self, initial_balance):
        self._balance = initial_balance

    @property
    def balance(self):
        return self._balance

    @balance.setter
    def balance(self, value):
        if value < 0:
            raise ValueError("Balance cannot be negative!")
        self._balance = value

acc = BankAccount(100)
acc.balance = 250  # Uses setter validation

print("Current Account Balance via @property:", acc.balance)


### Q56: What are Abstract Base Classes (abc.ABC, @abstractmethod) and how do they enforce Interface Contracts?

#### Verbal Answer:

Abstract Base Classes (`abc.ABC`) enforce interface compliance:

1. **Core Behavior**:
   * Classes containing `@abstractmethod` **cannot be directly instantiated**.
   * Any subclass **must implement all abstract methods**, or instantiation raises `TypeError: Can't instantiate abstract class... with abstract methods`.

2. **Virtual Subclasses (`ABC.register()`)**:
   * Allows third-party classes to satisfy `isinstance()` checks without inheriting directly.

---

#### Tricky Follow-up:

* **Question:** *Can an `@abstractmethod` contain a default implementation in the base class?*

* **Answer:** Yes. Subclasses can call `super().abstract_method()` to reuse base logic while still being required to override it.



In [ ]:
from abc import ABC, abstractmethod

class PaymentGateway(ABC):
    @abstractmethod
    def process_payment(self, amount: float) -> bool:
        pass

class StripeGateway(PaymentGateway):
    def process_payment(self, amount: float) -> bool:
        return True

stripe = StripeGateway()
print("1. StripeGateway instantiated successfully:", stripe.process_payment(100.0))
print("2. Is instance of PaymentGateway ABC:      ", isinstance(stripe, PaymentGateway))


### Q57: What is the difference between __getattr__ and __getattribute__ in Python?

#### Verbal Answer:

Attribute interception mechanics:

1. **`__getattr__(self, name)` (Fallback)**:
   * Called **ONLY when the attribute is NOT found** in the instance `__dict__` or class hierarchy.
   * Ideal for proxy patterns, lazy attribute loading, or mock objects.

2. **`__getattribute__(self, name)` (Universal Interceptor)**:
   * Called **unconditionally on EVERY attribute access** before looking in `__dict__`.
   * Extreme caution required: Accessing `self.attr` inside `__getattribute__` causes infinite recursion! Must delegate to `super().__getattribute__(name)`.

---

#### Tricky Follow-up:

* **Question:** *How do you prevent infinite recursion inside `__getattribute__`?*

* **Answer:** Use `super().__getattribute__(name)` or `object.__getattribute__(self, name)`.



In [ ]:
class DynamicProxy:
    def __init__(self):
        self.existing_field = "Real Value"

    def __getattr__(self, name):
        return f"Dynamically resolved property: '{name}'"

proxy = DynamicProxy()

print("1. Existing attribute (bypasses __getattr__):", proxy.existing_field)
print("2. Non-existent attribute (triggers __getattr__):", proxy.dynamic_calc)


### Q58: What are the core Operator Overloading dunder methods (__add__, __mul__, __eq__, __lt__, __call__)?

#### Verbal Answer:

Operator Overloading maps Python syntax to dunder methods:

1. **Arithmetic**:
   * `__add__(self, other)`: `+`
   * `__sub__(self, other)`: `-`
   * `__mul__(self, other)`: `*`
   * `__matmul__(self, other)`: Matrix multiplication `@`

2. **Comparisons**:
   * `__eq__`: `==`, `__ne__`: `!=`, `__lt__`: `<`, `__le__`: `<=`, `__gt__`: `>`, `__ge__`: `>=`

3. **Callable Object**:
   * `__call__(self, *args)`: Allows an instance to be invoked as a function `obj()`.

---

#### Tricky Follow-up:

* **Question:** *What is `functools.total_ordering` and how does it reduce boilerplate in comparison classes?*

* **Answer:** It automatically generates all comparison methods (`<=`, `>`, `>=`) when you define `__eq__` and any one ordering method (`__lt__`).



In [ ]:
class Vector2D:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __add__(self, other):
        return Vector2D(self.x + other.x, self.y + other.y)

    def __call__(self, scale):
        return Vector2D(self.x * scale, self.y * scale)

    def __repr__(self):
        return f"Vector2D({self.x}, {self.y})"

v1 = Vector2D(2, 3)
v2 = Vector2D(4, 5)
v3 = v1 + v2         # Calls __add__
v4 = v3(10)          # Calls __call__

print("1. Vector Addition (v1 + v2):", v3)
print("2. Callable Invocation (v3(10)):", v4)


### Q59: What is the difference between __new__() and __init__() in object lifecycle creation?

#### Verbal Answer:

Object Instantiation Lifecycle:

1. **`__new__(cls, *args, **kwargs)` (Allocator / Constructor)**:
   * Static method responsible for **allocating and returning the physical object instance** in memory.
   * First parameter is the class `cls`.
   * Must return an instance of `cls` for `__init__` to be called.
   * Used for subclassing immutable types (`int`, `str`, `tuple`), Metaprogramming, and Singleton patterns.

2. **`__init__(self, *args, **kwargs)` (Initializer)**:
   * Receives the allocated instance `self`.
   * Initializes instance attributes and state.

---

#### Tricky Follow-up:

* **Question:** *How can you implement the Singleton Pattern in Python using `__new__`?*

* **Answer:** Store `_instance` on the class, and in `__new__`, return `_instance` if it already exists rather than allocating a new object.



In [ ]:
class SingletonDatabase:
    _instance = None

    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

db1 = SingletonDatabase()
db2 = SingletonDatabase()

print("Singleton verification (db1 is db2):", db1 is db2)


### Q60: What is dataclasses.dataclass and how do frozen=True, order=True, and field(default_factory=...) work?

#### Verbal Answer:

`dataclasses` (PEP 557) automates boilerplate class creation:

1. **Automated Dunder Generation**:
   * Automatically generates `__init__()`, `__repr__()`, and `__eq__()` from type annotations.

2. **Configuration Options**:
   * `frozen=True`: Makes instances immutable and hashable.
   * `order=True`: Generates comparison dunder methods (`<`, `<=`, `>`, `>=`).
   * `field(default_factory=list)`: Prevents mutable default bugs for lists/dicts.
   * `field(repr=False)`: Excludes sensitive fields (e.g. passwords) from `__repr__`.

---

#### Tricky Follow-up:

* **Question:** *What post-initialization hook is available in a `@dataclass`?*

* **Answer:** `def __post_init__(self):` executes immediately after `__init__` for custom validation and derived field calculations.



In [ ]:
from dataclasses import dataclass, field

@dataclass(order=True)
class TradeOrder:
    priority: int
    symbol: str = field(compare=False)
    tags: list = field(default_factory=list, compare=False)

o1 = TradeOrder(priority=1, symbol='AAPL')
o2 = TradeOrder(priority=2, symbol='MSFT')

print("1. Dataclass Repr:              ", o1)
print("2. Ordered Comparison (o1 < o2):", o1 < o2)


## Category 7: Decorators & Metaprogramming Patterns


### Q61: What is a Decorator in Python and how does the @decorator syntactic sugar work under the hood?

#### Verbal Answer:

A **Decorator** is a higher-order function that takes another function as an argument, extends or modifies its behavior without modifying its source code, and returns the modified function:

Syntactic Sugar Equivalence:
```python
@my_decorator
def target_function(): ...
```
is completely equivalent to:
```python
target_function = my_decorator(target_function)
```
Executed at function definition time.

---

#### Tricky Follow-up:

* **Question:** *What happens if a decorator does not return a callable?*

* **Answer:** The decorated function name is rebound to whatever the decorator returned (e.g. `None` if return is omitted), causing a `TypeError: 'NoneType' object is not callable` when invoked.



In [ ]:
def debug_logger(func):
    def wrapper(*args, **kwargs):
        print(f"[LOG] Executing function: '{func.__name__}' with args={args}")
        result = func(*args, **kwargs)
        print(f"[LOG] Function '{func.__name__}' finished.")
        return result
    return wrapper

@debug_logger
def compute_area(width, height):
    return width * height

res = compute_area(10, 20)
print("Returned Calculation Result:", res)


### Q62: Why is functools.wraps mandatory when authoring custom decorators?

#### Verbal Answer:

`functools.wraps(func)` preserves function introspection metadata:

1. **The Problem Without `wraps`**:
   * Wrapping a function replaces it with `wrapper`.
   * The original function's `__name__`, `__doc__`, `__module__`, and `__annotations__` are lost (all report `'wrapper'`), breaking debuggers, documentation generators (Sphinx), and serializers.

2. **The Solution**:
   * `@functools.wraps(func)` copies all original metadata and binds `wrapper.__wrapped__ = func` for accessing the original unwrapped callable.

---

#### Tricky Follow-up:

* **Question:** *How can you bypass a decorator to invoke the original undecorated function directly?*

* **Answer:** Call `decorated_function.__wrapped__(*args, **kwargs)`.



In [ ]:
import functools

def audit_decorator(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        """Wrapper Docstring"""
        return func(*args, **kwargs)
    return wrapper

@audit_decorator
def execute_trade(ticker, qty):
    """Execute a buy/sell trade order."""
    return f"Order placed for {qty} shares of {ticker}"

print("1. Function Name preserved:     ", execute_trade.__name__)
print("2. Function Docstring preserved:", execute_trade.__doc__)
print("3. Unwrapped function access:   ", execute_trade.__wrapped__('AAPL', 100))


### Q63: How do you write Parameterized Decorators (Decorators with Arguments)?

#### Verbal Answer:

Parameterized Decorators require a **3-Tier Nested Function Architecture**:

1. **Outer Function**: Receives decorator configuration arguments (`@retry(max_attempts=3)`).
2. **Middle Function**: Receives the target function to decorate (`func`).
3. **Inner Wrapper Function**: Receives runtime arguments (`*args, **kwargs`) and executes the wrapped logic.

---

#### Tricky Follow-up:

* **Question:** *What is the order of execution when stacking multiple decorators on a single function?*

* **Answer:** Decorators execute in **Bottom-Up order** at definition time (`@dec1; @dec2; def f()` evaluates as `f = dec1(dec2(f))`).



In [ ]:
def repeat(times):
    def decorator(func):
        def wrapper(*args, **kwargs):
            results = []
            for _ in range(times):
                results.append(func(*args, **kwargs))
            return results
        return wrapper
    return decorator

@repeat(times=3)
def ping_server():
    return "PONG"

print("Parameterized Decorator Execution Output:", ping_server())


### Q64: How do you implement Class-Based Decorators using __call__()?

#### Verbal Answer:

Class-based decorators maintain stateful decorator configurations across function calls:

1. **Mechanism**:
   * `__init__(self, func)`: Captures the target function at definition time.
   * `__call__(self, *args, **kwargs)`: Executes each time the decorated function is invoked.

2. **Use Cases**:
   * Rate-limiting, call counting, distributed state tracking, and circuit breakers.

---

#### Tricky Follow-up:

* **Question:** *What issue arises when using a class-based decorator on an instance method inside a class?*

* **Answer:** The instance `self` is not bound automatically because the decorator instance does not implement the Descriptor Protocol. Must define `__get__()` to bind methods.



In [ ]:
class CallCounter:
    def __init__(self, func):
        self.func = func
        self.count = 0

    def __call__(self, *args, **kwargs):
        self.count += 1
        return self.func(*args, **kwargs)

@CallCounter
def process_data():
    return "OK"

process_data()
process_data()
process_data()

print("Class-Based Decorator State Tracking (Total Invocations):", process_data.count)


### Q65: How does functools.lru_cache work internally and how do maxsize and typed parameters affect caching?

#### Verbal Answer:

`functools.lru_cache(maxsize=128, typed=False)` provides Least-Recently-Used in-memory memoization:

1. **Parameters**:
   * `maxsize=128`: Maximum cached results. When full, evicts oldest least recently accessed item. If `maxsize=None`, cache grows without bound.
   * `typed=True`: Treats arguments of different types separately (e.g. `f(3)` and `f(3.0)` cached as distinct entries).

2. **Cache Management**:
   * `func.cache_info()`: Returns `CacheInfo(hits, misses, maxsize, currsize)`.
   * `func.cache_clear()`: Flushes all cached entries.

---

#### Tricky Follow-up:

* **Question:** *What constraint must all arguments to an `lru_cache` decorated function satisfy?*

* **Answer:** All arguments MUST be **hashable** (e.g. tuples, strings, ints). Passing mutable lists or dicts raises `TypeError: unhashable type`.



In [ ]:
import functools

@functools.lru_cache(maxsize=32)
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

res = fib(30)
cache_stats = fib.cache_info()

print("1. Fibonacci(30) Result:", res)
print("2. LRU Cache Diagnostics:", cache_stats)


### Q66: What is functools.singledispatch and how does it implement Method Overloading in Python?

#### Verbal Answer:

`@functools.singledispatch` implements **Generic Functions / Single-Dispatch Function Overloading** based on the type of the first argument:

1. **How It Operates**:
   * Defines a base fallback implementation.
   * `@func.register(int)` or `@func.register(list)` registers specialized implementations for specific types.
   * Python 3.7+ can infer the dispatch type directly from type annotations (`@func.register def _(arg: int): ...`).

---

#### Tricky Follow-up:

* **Question:** *What is the difference between single dispatch and multiple dispatch?*

* **Answer:** Single dispatch overloads strictly on the type of the **first** argument; Multiple dispatch overloads on the combined types of all arguments simultaneously.



In [ ]:
from functools import singledispatch

@singledispatch
def format_payload(arg):
    return f"Default string representation: {arg}"

@format_payload.register(int)
def _(arg: int):
    return f"Integer payload: {arg:,} (Hex: {hex(arg)})"

@format_payload.register(list)
def _(arg: list):
    return f"List payload containing {len(arg)} items: {arg}"

print("1. Integer dispatch:", format_payload(1000000))
print("2. List dispatch:   ", format_payload(['A', 'B', 'C']))
print("3. Default fallback:", format_payload({'key': 'val'}))


### Q67: How do you write a Decorator that can be called both with and without parentheses (@dec and @dec())?

#### Verbal Answer:

Dual-Call Decorator Architecture:

1. **The Pattern**:
   * If called without arguments (`@my_decorator`), the target function is passed directly as the first argument (`_func is not None`).
   * If called with arguments (`@my_decorator(param=val)`), `_func is None`, returning a decorator function that accepts the target function.

2. **Implementation**:
   ```python
   def universal_decorator(_func=None, *, custom_param=True):
       def decorator(func):
           @functools.wraps(func)
           def wrapper(*args, **kwargs):
               return func(*args, **kwargs)
           return wrapper
       return decorator(_func) if _func is not None else decorator
   ```

---

#### Tricky Follow-up:

* **Question:** *Why is this dual-call pattern essential in library APIs like Pytest fixtures or Flask routes?*

* **Answer:** To allow clean syntax for both `@pytest.fixture` and `@pytest.fixture(scope='session')` without forcing developers to remember parentheses.



In [ ]:
import functools

def smart_log(_func=None, *, tag="DEFAULT"):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            return f"[{tag}] {func(*args, **kwargs)}"
        return wrapper
    if _func is not None:
        return decorator(_func)
    return decorator

@smart_log
def greet(): return "Hello"

@smart_log(tag="CUSTOM")
def farewell(): return "Goodbye"

print("1. Decorated without parens:", greet())
print("2. Decorated with parens:   ", farewell())


### Q68: How do Class Decorators work and how do they modify or register entire class definitions?

#### Verbal Answer:

**Class Decorators** inspect, mutate, or wrap class definitions at definition time:

1. **Mechanism**:
   * Accepts a `cls` class object as its argument.
   * Can inject new methods, wrap existing methods, add `__repr__` (like `@dataclass`), or register the class in a central plugin registry.
   * Returns the modified class or a proxy wrapper.

---

#### Tricky Follow-up:

* **Question:** *How does a class decorator differ from a Metaclass?*

* **Answer:** Class decorators operate **after** the class has already been constructed by its metaclass; metaclasses intercept class creation **during** construction.



In [ ]:
def register_plugin(cls):
    cls._is_plugin = True
    cls.plugin_id = cls.__name__.lower()
    return cls

@register_plugin
class PaymentProcessor:
    def process(self): return "Processing"

print("Class Decorator Injected Metadata:")
print("Is Plugin:", getattr(PaymentProcessor, '_is_plugin', False))
print("Plugin ID:", PaymentProcessor.plugin_id)


### Q69: How do you implement a Retry Decorator with Exponential Backoff and Jitter?

#### Verbal Answer:

Production Fault-Tolerance Retry Pattern:

1. **Core Mechanics**:
   * Intercepts transient network or database exceptions.
   * Retries up to `max_retries` with exponentially increasing sleep delays: `delay = base_delay * (2 ** attempt)`.
   * Adds random **Jitter** to prevent the 'Thundering Herd' problem across distributed workers.

---

#### Tricky Follow-up:

* **Question:** *Why is random jitter necessary in distributed retry systems?*

* **Answer:** If 10,000 workers fail simultaneously, exponential backoff without jitter causes all 10,000 workers to retry at the exact same synchronized interval, crashing the recovering database again.



In [ ]:
import time
import random

def retry(max_attempts=3, base_delay=0.01):
    def decorator(func):
        def wrapper(*args, **kwargs):
            attempts = 0
            while attempts < max_attempts:
                try:
                    return func(*args, **kwargs)
                except ValueError as e:
                    attempts += 1
                    if attempts == max_attempts:
                        raise e
                    sleep_time = base_delay * (2 ** attempts) + random.uniform(0, 0.01)
                    time.sleep(sleep_time)
        return wrapper
    return decorator

call_count = 0
@retry(max_attempts=3)
def unstable_network_call():
    global call_count
    call_count += 1
    if call_count < 2:
        raise ValueError("Network Timeout")
    return "Network Success"

print("Retry Decorator Execution:", unstable_network_call())


### Q70: How do you create an Access Control & Authorization Decorator for API endpoints?

#### Verbal Answer:

Security & Access Control Enforcement via Decorators:

1. **Pattern**:
   * Inspects current execution context (user roles, tokens, permissions) before executing sensitive business logic.
   * Raises `PermissionError` or returns HTTP 403 Forbidden if authorization checks fail.

---

#### Tricky Follow-up:

* **Question:** *Where should input validation decorators be placed relative to authentication decorators in a decorator stack?*

* **Answer:** Authentication should be on the **outer/top layer** so unauthorized requests are rejected immediately before spending CPU cycles on input parsing.



In [ ]:
def require_roles(*allowed_roles):
    def decorator(func):
        def wrapper(user, *args, **kwargs):
            if user.get('role') not in allowed_roles:
                raise PermissionError(f"Access Denied for role: {user.get('role')}")
            return func(user, *args, **kwargs)
        return wrapper
    return decorator

@require_roles('Admin', 'SuperUser')
def delete_database_record(user, record_id):
    return f"Record {record_id} successfully deleted by {user['username']}."

admin_user = {'username': 'alice', 'role': 'Admin'}
print("Authorized Execution:", delete_database_record(admin_user, 999))


## Category 8: Context Managers & Resource Management


### Q71: How does the Context Management Protocol (__enter__ and __exit__) guarantee deterministic resource cleanup?

#### Verbal Answer:

The Context Management Protocol (`with` statement, PEP 343):

1. **`__enter__(self)`**:
   * Allocates resources (locks, socket, file descriptor).
   * Return value is bound to the optional `as target` variable.

2. **`__exit__(self, exc_type, exc_val, exc_tb)`**:
   * **Guaranteed execution** upon exiting the `with` block (even if uncaught exceptions, `return`, `break`, or `sys.exit` occur).
   * If an exception occurred, receives the exception details.
   * Returning `True` **suppresses** the exception; returning `False` or `None` propagates it upwards.

---

#### Tricky Follow-up:

* **Question:** *Under what circumstance would `__exit__()` intentionally return `True`?*

* **Answer:** When implementing an error suppression manager (e.g. `contextlib.suppress(FileNotFoundError)`) to swallow specific expected exceptions.



In [ ]:
class DatabaseConnection:
    def __enter__(self):
        print("1. [__enter__] Database connection opened.")
        return self

    def query(self, sql):
        return f"Results for '{sql}'"

    def __exit__(self, exc_type, exc_val, exc_tb):
        print("3. [__exit__] Database connection closed cleanly.")
        return False

with DatabaseConnection() as db:
    print("2. Executing query:", db.query("SELECT * FROM users"))


### Q72: What are contextlib.ExitStack and AsyncExitStack and why are they essential for managing dynamic numbers of context managers?

#### Verbal Answer:

`contextlib.ExitStack` manages dynamic or programmatic collections of context managers:

1. **The Limitation of Static `with`**:
   * `with open('a') as f1, open('b') as f2:` requires knowing the exact file count at compile time.

2. **ExitStack Solution**:
   * Programmatically enters an arbitrary runtime list of context managers.
   * Guarantees LIFO (Last-In, First-Out) cleanup of all entered managers if any error occurs during setup.

---

#### Tricky Follow-up:

* **Question:** *How does `ExitStack.callback(func, *args)` allow registering arbitrary cleanup functions?*

* **Answer:** It registers any arbitrary callable to be executed upon block exit, acting like a programmable `defer` statement.



In [ ]:
from contextlib import ExitStack
import tempfile, os

with ExitStack() as stack:
    # Programmatically open dynamic temporary files
    temp_files = [stack.enter_context(tempfile.NamedTemporaryFile(mode='w+')) for _ in range(3)]
    for idx, tf in enumerate(temp_files):
        tf.write(f"Data stream {idx}")
        tf.flush()
    print("Successfully wrote to 3 dynamically managed files using ExitStack.")

print("All dynamic resources cleanly cleaned up.")


### Q73: How does contextlib.suppress(*exceptions) replace verbose try-except-pass blocks?

#### Verbal Answer:

`contextlib.suppress()` cleanly silences specific expected exceptions:

1. **Cleaner Alternative to**:
   ```python
   try:
       os.remove('temp.txt')
   except FileNotFoundError:
       pass
   ```
2. **Pythonic Idiom**:
   ```python
   with contextlib.suppress(FileNotFoundError):
       os.remove('temp.txt')
   ```

---

#### Tricky Follow-up:

* **Question:** *Can `contextlib.suppress()` suppress multiple exception types simultaneously?*

* **Answer:** Yes: `with suppress(FileNotFoundError, PermissionError, KeyError): ...`.



In [ ]:
import contextlib
import os

# Safe deletion without try-except-pass boilerplate
with contextlib.suppress(FileNotFoundError):
    os.remove('non_existent_file_to_delete.txt')

print("contextlib.suppress cleanly handled non-existent file.")


### Q74: How do contextlib.redirect_stdout and redirect_stderr capture or redirect standard stream outputs?

#### Verbal Answer:

Redirecting system stdout / stderr streams:

1. **Mechanism**:
   * Temporarily rebinds `sys.stdout` or `sys.stderr` to a stream buffer (e.g. `io.StringIO()`) for the duration of the `with` block.
   * Restores original console stream unconditionally upon exit.

---

#### Tricky Follow-up:

* **Question:** *Can `redirect_stdout` capture C-level `printf` output from external C-extensions?*

* **Answer:** No. `redirect_stdout` only rebinds Python's `sys.stdout`. Capturing C-level OS file descriptor 1 requires operating system level `os.dup2()` pipe redirection.



In [ ]:
import contextlib
import io

buffer = io.StringIO()
with contextlib.redirect_stdout(buffer):
    print("This text was captured in-memory!")

captured_content = buffer.getvalue()
print("Captured stdout output:", captured_content.strip())


### Q75: How do you implement a Reentrant and Thread-Safe Context Manager (threading.RLock)?

#### Verbal Answer:

Reentrant Context Managers:

1. **Reentrant Lock (`threading.RLock`)**:
   * Can be acquired multiple times by the **same thread** without deadlocking.
   * Tracks recursion level and is released when the outermost `with` block exits.

2. **Custom Reentrant Context Manager**:
   * Maintains internal reference counter incremented in `__enter__` and decremented in `__exit__`.

---

#### Tricky Follow-up:

* **Question:** *What happens if a thread tries to re-acquire a standard `threading.Lock` inside a nested function call?*

* **Answer:** The thread deadlocks with itself because `threading.Lock` can only be acquired once.



In [ ]:
import threading

rlock = threading.RLock()

def nested_computation():
    with rlock:  # Re-enters safely without deadlocking
        return "Inner computation finished"

with rlock:
    res = nested_computation()

print("Reentrant RLock Nested Execution Result:", res)


### Q76: What is contextlib.nullcontext and when is it used as an optional context manager?

#### Verbal Answer:

`contextlib.nullcontext(enter_result=None)` is a **no-op dummy context manager**:

1. **Primary Use Case**:
   * Handling conditional resource locks cleanly:
   ```python
   cm = my_lock if enable_locking else nullcontext()
   with cm:
       critical_section()
   ```
   * Eliminates duplicate `if-else` blocks.

---

#### Tricky Follow-up:

* **Question:** *Can `nullcontext` return a predefined value when entered?*

* **Answer:** Yes. `with nullcontext(default_val) as val:` binds `val = default_val`.



In [ ]:
from contextlib import nullcontext

def process_pipeline(custom_stream=None):
    # If custom_stream is None, use nullcontext
    with (custom_stream or nullcontext("Default String Buffer")) as stream:
        return f"Processed: {stream}"

print("1. With default stream:", process_pipeline())


### Q77: How do you implement an Atomic Transaction Rollback Context Manager for In-Memory Dictionaries?

#### Verbal Answer:

Atomic Rollback Context Manager Pattern:

1. **Mechanism**:
   * In `__enter__`, create a deep copy / snapshot of the dictionary state.
   * In `__exit__`, if `exc_type is not None`, rollback the dictionary to the initial snapshot and re-raise or suppress the error.

---

#### Tricky Follow-up:

* **Question:** *Why is this atomic snapshot pattern critical in financial transaction ledgers?*

* **Answer:** To ensure ACID atomicity: multi-step account updates either succeed completely or leave account balances completely unaltered if an error occurs halfway.



In [ ]:
import copy

class AtomicDictionary:
    def __init__(self, data):
        self.data = data

    def __enter__(self):
        self.snapshot = copy.deepcopy(self.data)
        return self.data

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is not None:
            # Rollback to snapshot on error
            self.data.clear()
            self.data.update(self.snapshot)
            print("[ROLLBACK] Exception caught! Restored original ledger.")
            return True  # Suppress for demo

ledger = {'Alice': 500, 'Bob': 200}

with AtomicDictionary(ledger) as l:
    l['Alice'] -= 100
    l['Bob'] += 100
    raise RuntimeError("Payment Network Failure!")

print("Final Ledger Balance after Rollback:", ledger)


### Q78: How do Context Managers manage database sessions and transactions in SQLAlchemy / SQLite?

#### Verbal Answer:

Database Session Lifecycle Management:

1. **Transaction Lifecycle**:
   * `__enter__`: Opens session.
   * `__exit__`: Calls `session.commit()` on normal exit, or `session.rollback()` and closes session if an exception occurs.

---

#### Tricky Follow-up:

* **Question:** *What happens if a database connection is not closed inside `__exit__`?*

* **Answer:** Connection pool leakage occurs; exhausted connection pools cause incoming queries to stall and fail.



In [ ]:
import sqlite3
from contextlib import contextmanager

@contextmanager
def sqlite_transaction(db_name=":memory:"):
    conn = sqlite3.connect(db_name)
    try:
        yield conn.cursor()
        conn.commit()
    except Exception as e:
        conn.rollback()
        raise e
    finally:
        conn.close()

with sqlite_transaction() as cursor:
    cursor.execute("CREATE TABLE accounts (id INT, balance INT)")
    cursor.execute("INSERT INTO accounts VALUES (1, 1000)")

print("Database transaction committed and connection closed cleanly.")


### Q79: What are Async Context Managers (async with, __aenter__, __aexit__)?

#### Verbal Answer:

Asynchronous Context Managers (PEP 492):

1. **Protocol**:
   * `async def __aenter__(self)`: Awaited upon entering `async with`.
   * `async def __aexit__(self, exc_type, exc_val, exc_tb)`: Awaited upon exit for async network/socket cleanup.

2. **Decorator Helper**:
   * `@contextlib.asynccontextmanager` allows authoring async context managers with `async def` generators.

---

#### Tricky Follow-up:

* **Question:** *Can you use a standard synchronous context manager inside an `async def` function?*

* **Answer:** Yes, using standard `with obj:`, as long as the cleanup operations do not block the event loop with synchronous I/O.



In [ ]:
import asyncio

class AsyncDatabaseMock:
    async def __aenter__(self):
        await asyncio.sleep(0.01)  # Async setup
        return self

    async def fetch(self):
        return "Async Record Data"

    async def __aexit__(self, exc_type, exc_val, exc_tb):
        await asyncio.sleep(0.01)  # Async cleanup

async def main():
    async with AsyncDatabaseMock() as db:
        res = await db.fetch()
        print("Async Context Manager Result:", res)

asyncio.run(main())


### Q80: What is the difference between finalizers, weak references (weakref.finalize), and __del__ destructors?

#### Verbal Answer:

Object Cleanup & Destructors:

1. **`__del__()` (Prone to Gotchas)**:
   * Called when reference count reaches 0.
   * Problematic with cyclic references and module shutdown ordering.

2. **`weakref.finalize(obj, cleanup_func, *args)` (Modern Best Practice)**:
   * Registers a cleanup callback that runs **when `obj` is garbage collected**, without keeping `obj` alive.
   * Immune to cyclic reference destructor blocking.

---

#### Tricky Follow-up:

* **Question:** *Why should you never pass `self` as an argument to `weakref.finalize(self, self.cleanup)`?*

* **Answer:** Passing `self` creates a strong reference inside the finalizer, preventing the object from ever reaching refcount 0!



In [ ]:
import weakref

class ResourceHolder:
    def __init__(self, name):
        self.name = name
        # Correct: Pass a standalone function or bound resource handle
        weakref.finalize(self, print, f"[CLEANUP] ResourceHolder '{self.name}' finalized.")

res = ResourceHolder("Socket_9000")
del res  # Triggers weakref finalizer immediately


## Category 9: Error Handling, Custom Exceptions & Defensive Programming


### Q81: How does Python's Exception Hierarchy work and why should you never catch bare 'except:' or 'except Exception:' indiscriminately?

#### Verbal Answer:

Python Built-in Exception Hierarchy:

1. **Hierarchy Tree**:
   * `BaseException`
     * `SystemExit` (triggered by `sys.exit()`)
     * `KeyboardInterrupt` (`Ctrl+C`)
     * `GeneratorExit`
     * `Exception` (Base for all standard program errors: `ValueError`, `KeyError`, `TypeError`)

2. **The Danger of Bare `except:`**:
   * Catching bare `except:` or `except BaseException:` catches `KeyboardInterrupt` and `SystemExit`, making programs **impossible to terminate with Ctrl+C or kill signals**!
   * Always catch specific exceptions: `except (KeyError, ValueError):`.

---

#### Tricky Follow-up:

* **Question:** *Why is `except Exception:` preferred over bare `except:` if a broad catch is necessary?*

* **Answer:** Because `Exception` catches standard errors while still allowing `KeyboardInterrupt` and `SystemExit` to terminate the process cleanly.



In [ ]:
try:
    int("invalid_number")
except ValueError as e:
    print("1. Specifically caught expected ValueError:", e)

print("2. Exception hierarchy: issubclass(ValueError, Exception) ->", issubclass(ValueError, Exception))


### Q82: What is Exception Chaining (raise ... from ...) and how does it differentiate explicit vs implicit chaining?

#### Verbal Answer:

Exception Chaining (PEP 3134):

1. **Explicit Chaining (`raise NewError(...) from original_error`)**:
   * Sets `__cause__` attribute.
   * Cleanly transforms low-level exceptions into high-level domain exceptions (e.g. `sqlite3.OperationalError` into `DatabaseConnectionError`).

2. **Implicit Chaining**:
   * If an exception occurs while handling another exception in an `except` block, Python sets `__context__` ("During handling of the above exception, another exception occurred").

3. **Suppressing Chaining**:
   * `raise NewError() from None` hides low-level internal stack trace details from end users.

---

#### Tricky Follow-up:

* **Question:** *When should you use `raise MyDomainError from None`?*

* **Answer:** When writing clean user-facing APIs where internal implementation exceptions (like private dict lookups) are irrelevant noise to the caller.



In [ ]:
class APIRequestError(Exception): pass

def fetch_data():
    try:
        int("corrupt_network_payload")
    except ValueError as e:
        # Explicit exception chaining
        raise APIRequestError("Failed to parse API payload") from e

try:
    fetch_data()
except APIRequestError as err:
    print("Caught Chained Error:", err)
    print("Original Root Cause:", err.__cause__)


### Q83: What are Exception Groups (ExceptionGroup) and except* syntax introduced in Python 3.11 (PEP 654)?

#### Verbal Answer:

Exception Groups & `except*` Syntax (Python 3.11+):

1. **ExceptionGroup**:
   * Bundles multiple unrelated concurrent exceptions into a single composite exception object.

2. **`except*` Syntax**:
   * Matches and handles specific exception subtypes within an `ExceptionGroup` **concurrently**, allowing other unhandled exceptions in the group to propagate.

---

#### Tricky Follow-up:

* **Question:** *Where are Exception Groups primarily used in modern Python?*

* **Answer:** In `asyncio.TaskGroup` for managing concurrent task failures simultaneously.



In [ ]:
try:
    raise ExceptionGroup("Batch Job Failures", [
        ValueError("Invalid format"),
        TypeError("Type mismatch"),
        ValueError("Negative value")
    ])
except* ValueError as eg:
    print("Caught all ValueErrors in group:", eg.exceptions)
except* TypeError as eg:
    print("Caught all TypeErrors in group: ", eg.exceptions)


### Q84: How do you design a Production-Grade Domain Exception Hierarchy for enterprise libraries?

#### Verbal Answer:

Enterprise Domain Exception Architecture:

1. **Base Domain Exception**:
   * Define a root exception for your package: `class AppBaseError(Exception): pass`.

2. **Hierarchical Categorization**:
   * Subdivide by domain subsystems:
     * `class ValidationError(AppBaseError): pass`
     * `class AuthenticationError(AppBaseError): pass`
     * `class RateLimitExceededError(AppBaseError): pass`

3. **Benefit to Callers**:
   * Users can catch all library errors with `except AppBaseError:` or catch fine-grained errors specifically.

---

#### Tricky Follow-up:

* **Question:** *Should custom exceptions inherit directly from `BaseException` or `Exception`?*

* **Answer:** Always inherit from `Exception`. Inheriting from `BaseException` bypasses standard `except Exception:` handlers.



In [ ]:
class FinTechAppError(Exception): """Base package error."""
class InsufficientFundsError(FinTechAppError): """Raised when balance < withdrawal."""

def withdraw(balance, amount):
    if amount > balance:
        raise InsufficientFundsError(f"Cannot withdraw ${amount} from balance ${balance}")
    return balance - amount

try:
    withdraw(100, 500)
except FinTechAppError as e:
    print("Caught Domain Specific Exception:", e)


### Q85: What is the traceback module and how do you format stack traces into structured JSON logs?

#### Verbal Answer:

Extracting structured stack traces via the `traceback` module:

1. **Key Functions**:
   * `traceback.format_exc()`: Returns the full stack trace string.
   * `traceback.extract_tb(tb)`: Returns structured `FrameSummary` objects with filename, line number, and function name.

2. **Structured JSON Logging**:
   * Critical in production microservices (Datadog/ELK/CloudWatch) to log error type, message, and call stack in JSON fields without mangling logs.

---

#### Tricky Follow-up:

* **Question:** *How can you get the current stack trace without raising an exception?*

* **Answer:** Use `traceback.extract_stack()` or `traceback.print_stack()`.



In [ ]:
import traceback
import json

def buggy_function():
    return 1 / 0

try:
    buggy_function()
except ZeroDivisionError as e:
    tb_frames = traceback.extract_tb(e.__traceback__)
    last_frame = tb_frames[-1]
    log_entry = {
        'error_type': e.__class__.__name__,
        'error_message': str(e),
        'file': last_frame.filename.split('/')[-1],
        'line': last_frame.lineno,
        'func': last_frame.name
    }
    print("Structured Error Log Entry:", json.dumps(log_entry))


### Q86: How does sys.excepthook enable Global Uncaught Exception Handling and crash telemetry?

#### Verbal Answer:

Global Crash Telemetry with `sys.excepthook`:

1. **Mechanism**:
   * When an uncaught exception bubbles up to top level, Python delegates to `sys.excepthook(exc_type, exc_value, exc_traceback)`.

2. **Production Use Cases**:
   * Centralized crash reporting to Sentry, PagerDuty, or Slack.
   * Creating memory heap dumps or flushing open telemetry buffers before termination.

---

#### Tricky Follow-up:

* **Question:** *How can you restore the default standard Python crash handler?*

* **Answer:** Assign `sys.excepthook = sys.__excepthook__`.



In [ ]:
import sys

def custom_global_handler(exc_type, exc_val, exc_tb):
    print(f"[GLOBAL TELEMETRY HOOK] Uncaught {exc_type.__name__}: {exc_val}")

# Register custom global crash handler
original_hook = sys.excepthook
sys.excepthook = custom_global_handler

# Test invoking hook
sys.excepthook(KeyError, KeyError("MISSING_API_KEY"), None)

# Restore default hook
sys.excepthook = original_hook


### Q87: What is EAFP (Easier to Ask for Forgiveness than Permission) vs LBYL (Look Before You Leap) in Python?

#### Verbal Answer:

Pythonic Error Handling Philosophy:

1. **EAFP (Easier to Ask for Forgiveness than Permission - Python Standard)**:
   * Assume valid state and catch exceptions if it fails:
     ```python
     try:
         return d[key]
     except KeyError:
         return default
     ```
   * Faster on success paths; prevents Race Conditions (TOCTOU).

2. **LBYL (Look Before You Leap)**:
   * Explicitly verify conditions before acting: `if key in d: return d[key]`.
   * Prone to Race Conditions (e.g. `if os.path.exists(f): open(f)` -> file could be deleted by another process between `exists` check and `open`).

---

#### Tricky Follow-up:

* **Question:** *What does TOCTOU stand for in race condition security vulnerabilities?*

* **Answer:** Time-Of-Check to Time-Of-Use race condition.



In [ ]:
# EAFP dictionary access
data = {'active': True}

# EAFP pattern
try:
    status = data['active']
except KeyError:
    status = False

print("EAFP retrieved status successfully:", status)


### Q88: How do warnings (warnings.warn, filterwarnings) differ from exceptions?

#### Verbal Answer:

Warnings vs Exceptions in Python:

1. **Key Difference**:
   * **Exceptions**: Fatal interruptions that halt program flow unless explicitly caught.
   * **Warnings (`warnings.warn`)**: Informational alerts informing developers of deprecations, runtime inefficiency, or sub-optimal configs without stopping execution.

2. **Categories**:
   * `DeprecationWarning`, `UserWarning`, `FutureWarning`, `ResourceWarning`.

3. **Filtering**:
   * `warnings.filterwarnings('ignore')` or `warnings.filterwarnings('error')` (turns warnings into test-failing exceptions in CI/CD pipelines).

---

#### Tricky Follow-up:

* **Question:** *How do you treat all warnings as errors in Pytest test suites?*

* **Answer:** Add `-W error` flag to pytest or configure `filterwarnings = ['error']` in `pytest.ini`.



In [ ]:
import warnings

def legacy_api():
    warnings.warn(
        "legacy_api() is deprecated and will be removed in v3.0. Use modern_api().",
        DeprecationWarning,
        stacklevel=2
    )
    return "Legacy Data"

# Test warning
with warnings.catch_warnings(record=True) as captured_warnings:
    warnings.simplefilter("always")
    res = legacy_api()
    print("1. Function completed normally:", res)
    print("2. Warning captured cleanly:   ", captured_warnings[0].message)


### Q89: Why should assert statements NEVER be used for production data validation and security checks?

#### Verbal Answer:

The Critical `assert` Vulnerability:

1. **The Fatal Flaw**:
   * Python bytecode compiler **completely removes all `assert` statements** when running with optimization flags (`python -O` or `PYTHONOPTIMIZE=1`).

2. **Security Danger**:
   * If you write `assert user.is_admin`, running in optimized production mode silently deletes the assertion, granting unauthorized access to any user!

3. **Defensive Standard**:
   * Use `assert` strictly for internal unit tests and invariants.
   * For production logic, always write explicit `if not condition: raise ValidationError(...)`.

---

#### Tricky Follow-up:

* **Question:** *What exception does `assert condition, 'Message'` raise when it fails?*

* **Answer:** `AssertionError`.



In [ ]:
# Defensive validation pattern
def process_withdrawal(user, amount):
    # CORRECT: Explicit check that cannot be stripped by python -O
    if amount <= 0:
        raise ValueError("Withdrawal amount must be strictly positive!")
    return f"Processed ${amount} withdrawal."

print("Validated Execution Result:", process_withdrawal({'id': 1}, 100))


### Q90: How does the finally block execute during return, break, or unexpected exceptions?

#### Verbal Answer:

The Inviolable `finally` Guarantee:

1. **Guaranteed Execution**:
   * The `finally` block executes **unconditionally** before leaving the `try-except` structure, even if `return`, `break`, `continue`, or an unhandled exception occurred!

2. **Return in Finally Gotcha**:
   * If `finally` contains an explicit `return` statement, it **overrides and silences** any `return` or raised exception from the `try` block! Never put `return` inside `finally`.

---

#### Tricky Follow-up:

* **Question:** *What happens if a function executes `return 1` in `try` and `return 2` in `finally`?*

* **Answer:** The function returns `2`. The return in `finally` overwrites the return value from `try`.



In [ ]:
def test_finally_behavior():
    try:
        return "TRY_RETURN"
    finally:
        print("[FINALLY] Cleanup executed even after return statement!")

res = test_finally_behavior()
print("Function Result:", res)


## Category 10: Standard Library Superpowers


### Q91: How does the bisect module perform Binary Search and insertion on sorted sequences?

#### Verbal Answer:

`bisect` provides fast C-level Binary Search (O(log N)) on sorted lists:

1. **Key Functions**:
   * `bisect_left(sorted_list, x)`: Returns insertion point for `x` before existing entries.
   * `bisect_right(sorted_list, x)`: Returns insertion point after existing entries.
   * `insort(sorted_list, x)`: Inserts `x` maintaining sorted order.

2. **Primary Use Case**:
   * Grade / Tax bracket lookups, interval discretization, and priority queues.

---

#### Tricky Follow-up:

* **Question:** *What is the time complexity of `bisect.insort()` into a Python list?*

* **Answer:** O(N) because although finding the position is O(log N), inserting an element into a dynamic array requires O(N) memory shifting.



In [ ]:
import bisect

# Discretizing numeric scores into letter grades via bisect
breakpoints = [60, 70, 80, 90]
grades = ['F', 'D', 'C', 'B', 'A']

def score_to_grade(score):
    i = bisect.bisect_right(breakpoints, score)
    return grades[i]

scores = [55, 72, 85, 99]
computed_grades = [score_to_grade(s) for s in scores]

print("1. Scores: ", scores)
print("2. Grades Calculated via bisect:", computed_grades)


### Q92: How does the secrets module generate cryptographically secure tokens and passwords compared to random?

#### Verbal Answer:

`secrets` vs `random`:

1. **`random` Module (Insecure)**:
   * Uses **Mersenne Twister PRNG** (period 2^19937 - 1).
   * Fast, but **completely deterministic and predictable**. Observing 624 consecutive outputs allows an attacker to predict all future outputs!
   * NEVER use for passwords, session tokens, or API keys.

2. **`secrets` Module (CSPRNG)**:
   * Uses OS-level entropy sources (`/dev/urandom` on Unix, `CryptGenRandom` on Windows).
   * Cryptographically secure for session IDs, password reset tokens, and salts.

---

#### Tricky Follow-up:

* **Question:** *How do you generate a secure URL-safe token in one line?*

* **Answer:** Use `secrets.token_urlsafe(32)`.



In [ ]:
import secrets

# Generate cryptographically secure reset token and hex key
secure_token = secrets.token_urlsafe(16)
secure_hex = secrets.token_hex(16)

print("1. Cryptographically Secure Token:", secure_token)
print("2. Secure Hex Key:                ", secure_hex)


### Q93: What is the enum module (Enum, IntEnum, Flag, auto()) and why is it superior to string constants?

#### Verbal Answer:

`enum.Enum` creates type-safe, semantic enumerations:

1. **Key Types**:
   * `Enum`: Standard enumeration.
   * `IntEnum`: Subclass of `int` for integer comparisons.
   * `Flag` / `IntFlag`: Bitwise bitmask flags (`Permissions.READ | Permissions.WRITE`).
   * `auto()`: Automatically assigns sequential values.

2. **Benefits**:
   * Prevents typos, enables static typing, supports iteration and reverse lookup.

---

#### Tricky Follow-up:

* **Question:** *What decorator ensures all enum values are strictly unique without duplicates?*

* **Answer:** Use `@enum.unique`.



In [ ]:
from enum import Enum, Flag, auto

class OrderStatus(Enum):
    PENDING = auto()
    PROCESSING = auto()
    COMPLETED = auto()

class Permissions(Flag):
    READ = auto()
    WRITE = auto()
    EXECUTE = auto()

user_perms = Permissions.READ | Permissions.WRITE

print("1. Enum Member Name:", OrderStatus.COMPLETED.name, "| Value:", OrderStatus.COMPLETED.value)
print("2. Bitwise Flag Check (Has WRITE?):", bool(user_perms & Permissions.WRITE))


### Q94: How does the array module provide compact C-style homogeneous arrays in pure Python?

#### Verbal Answer:

`array.array(typecode, [initializer])` stores raw C-style arrays:

1. **Memory Compactness**:
   * Stores raw binary numbers without Python object pointer wrapping (e.g. `array('i')` stores 4-byte C integers, `array('d')` stores 8-byte C doubles).

2. **When to Use**:
   * When you need compact flat numeric buffers in pure Python without installing third-party NumPy.

---

#### Tricky Follow-up:

* **Question:** *Can a Python `array.array` object be converted directly to bytes for network transmission?*

* **Answer:** Yes, call `arr.tobytes()` and `arr.frombytes()`.



In [ ]:
import array
import sys

# Compact C-style integer array ('i' = signed 32-bit int)
c_arr = array.array('i', range(1000))
py_list = list(range(1000))

print("1. array.array buffer memory: ", sys.getsizeof(c_arr), "bytes")
print("2. Python list pointer memory:", sys.getsizeof(py_list), "bytes")
print("Memory Reduction Ratio:       ", round(sys.getsizeof(py_list) / sys.getsizeof(c_arr), 1), "x smaller!")


### Q95: How does copyreg allow customizing Pickle serialization for complex or C-level objects?

#### Verbal Answer:

`copyreg` registers custom pickling and unpickling functions:

1. **Pickle Customization**:
   * When standard `pickle` cannot serialize complex objects (open connections, C-pointers, thread locks), `copyreg.pickle()` registers a reducer callable returning reconstruction parameters.

2. **Security Rule**:
   * Never unpickle untrusted data from network sockets (leads to arbitrary remote code execution via `__reduce__`).

---

#### Tricky Follow-up:

* **Question:** *What method on a custom class overrides its pickling behavior directly without using `copyreg`?*

* **Answer:** Define the `__reduce__()` dunder method on the class.



In [ ]:
import pickle

# Pickle serialization with copyreg/reduce demonstration
data_payload = {'endpoint': 'https://api.com', 'timeout': 30, 'retries': 3}
pickled_bytes = pickle.dumps(data_payload)
restored_payload = pickle.loads(pickled_bytes)

print("Pickle serialization and reconstruction succeeded:", restored_payload['endpoint'], restored_payload['timeout'])


### Q96: How does graphlib.TopologicalSorter perform Dependency Resolution and cycle detection (DAGs)?

#### Verbal Answer:

`graphlib.TopologicalSorter` (Python 3.9+) solves **Directed Acyclic Graph (DAG)** dependencies:

1. **Core Capabilities**:
   * Resolves build system dependencies, task pipeline execution order, and package installation graphs.
   * Raises `CycleError` if circular dependencies exist.

2. **API**:
   * `ts = TopologicalSorter(graph)`
   * `tuple(ts.static_order())` returns valid linear topological order.

---

#### Tricky Follow-up:

* **Question:** *How can you execute independent tasks in parallel using `TopologicalSorter`?*

* **Answer:** Use the dynamic API: `ts.prepare()`, query `ts.get_ready()`, launch tasks, and mark them finished with `ts.done(task)`.



In [ ]:
from graphlib import TopologicalSorter

# Define DAG: {task: [dependencies]}
dependencies = {
    'deploy_prod': {'run_tests', 'build_docker'},
    'run_tests': {'lint_code'},
    'build_docker': {'lint_code'},
    'lint_code': set()
}

ts = TopologicalSorter(dependencies)
execution_order = list(ts.static_order())

print("Topological DAG Execution Order:", execution_order)


### Q97: How does the tempfile module (TemporaryFile, NamedTemporaryFile, TemporaryDirectory) safely handle scratch data?

#### Verbal Answer:

`tempfile` provides secure, auto-cleaning scratch storage:

1. **`TemporaryDirectory()`**:
   * Creates a dedicated scratch directory and auto-deletes it upon `with` exit.

2. **`NamedTemporaryFile(delete=True)`**:
   * Creates a file with a filesystem path accessible to other sub-processes, deleting on close.

3. **`TemporaryFile()`**:
   * Creates an anonymous OS file descriptor with no directory entry (cannot be accessed by other processes).

---

#### Tricky Follow-up:

* **Question:** *Why does `NamedTemporaryFile` on Windows cause `PermissionError` if reopened by another process while still open?*

* **Answer:** On Windows, open files are locked by default. In Python 3.12+, use `delete_on_close=False` to allow multi-process reopening.



In [ ]:
import tempfile
import os

with tempfile.TemporaryDirectory() as tmp_dir:
    temp_path = os.path.join(tmp_dir, "scratch.txt")
    with open(temp_path, "w") as f:
        f.write("Ephemeral compute payload")
    
    print("1. Temporary file created at:", os.path.basename(temp_path))
    print("2. File exists in directory: ", os.path.exists(temp_path))

print("3. Directory and file auto-deleted upon exit:", not os.path.exists(tmp_dir))


### Q98: How does the inspect module inspect stack frames, signatures, and source code dynamically?

#### Verbal Answer:

`inspect` module runtime introspection capabilities:

1. **`inspect.signature(func)`**:
   * Inspects parameters, default values, and type annotations.

2. **`inspect.currentframe()` & `stack()`**:
   * Traverses active call frames, caller names, and local variables.

3. **`inspect.getsource(func)`**:
   * Retrieves the raw Python source code string of a function.

---

#### Tricky Follow-up:

* **Question:** *How can you programmatically bind and validate arguments against a function's signature before executing it?*

* **Answer:** Use `sig.bind(*args, **kwargs)` which raises `TypeError` if arguments violate the function's parameter signature.



In [ ]:
import inspect

def target_pipeline(x: int, y: str = "default") -> bool:
    """Docstring demo"""
    return True

sig = inspect.signature(target_pipeline)
bound_args = sig.bind(42, y="custom")

print("1. Function Signature:   ", sig)
print("2. Bound Arguments Dict:", bound_args.arguments)
print("3. Return Annotation:   ", sig.return_annotation)


### Q99: How does the mmap module allow memory-mapped file access across Python processes?

#### Verbal Answer:

`mmap` memory mapping in Python:

1. **Mechanism**:
   * Maps a file on disk directly into the process virtual address space.
   * Access file contents using standard slicing syntax (`mm[0:100]`) with zero copying into Python user space buffers.

2. **Inter-Process Communication (IPC)**:
   * Multiple processes mapping the same file can communicate via shared memory pages at hardware speeds.

---

#### Tricky Follow-up:

* **Question:** *How do you create an anonymous memory map in RAM without backing it by a physical file on disk?*

* **Answer:** Pass `fileno=-1` and `length=size` to `mmap.mmap(-1, 1024)`.



In [ ]:
import mmap

# Anonymous in-memory shared buffer
mm = mmap.mmap(-1, 128)
mm.write(b"Hello Shared Memory Stream")
mm.seek(0)
read_data = mm.read(26)
mm.close()

print("Memory-Mapped Buffer Content:", read_data.decode('utf-8'))


### Q100: How does the zoneinfo module provide IANA timezone support in Python 3.9+ (PEP 615)?

#### Verbal Answer:

`zoneinfo.ZoneInfo` standardizes IANA timezones in the Python Standard Library:

1. **Modern Standard**:
   * Replaced third-party `pytz` in Python 3.9+.
   * Uses system timezone database or `tzdata` package.

2. **Daylight Saving Time Handling**:
   * Handles DST transitions accurately with standard `datetime.astimezone()` without requiring `pytz.localize()`.

---

#### Tricky Follow-up:

* **Question:** *What is the `fold` attribute on a `datetime` object?*

* **Answer:** `fold=0` or `fold=1` disambiguates wall-clock times that occur twice during the autumn Daylight Saving Time 'fall back' transition hour.



In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

# Create timezone-aware datetime in New York
dt_ny = datetime(2023, 10, 2, 9, 30, tzinfo=ZoneInfo("America/New_York"))

# Convert to Tokyo time
dt_tokyo = dt_ny.astimezone(ZoneInfo("Asia/Tokyo"))

print("1. New York Time:", dt_ny.strftime('%Y-%m-%d %H:%M:%S %Z'))
print("2. Tokyo Time:   ", dt_tokyo.strftime('%Y-%m-%d %H:%M:%S %Z'))


## Category 11: Python Concurrency: Threading, Multiprocessing & GIL


### Q101: What are the key trade-offs between Threading, Multiprocessing, and Asyncio in Python?

#### Verbal Answer:

Concurrency Paradigms Comparison Matrix:

1. **`threading` (Multi-Threading)**:
   * Architecture: Native OS threads sharing 1 process memory space; bound by GIL.
   * Best for: **I/O-bound tasks** (network requests, database calls, file reads).
   * Overheads: Low memory, shared memory state.

2. **`multiprocessing` (Multi-Processing)**:
   * Architecture: Separate OS processes, each with its own CPython interpreter, GIL, and isolated memory.
   * Best for: **CPU-bound tasks** (math, image processing, ML feature extraction).
   * Overheads: High memory, IPC serialization (pickle) overhead.

3. **`asyncio` (Cooperative Single-Threaded Asynchronous)**:
   * Architecture: Single thread, single event loop switching cooperatively on `await`.
   * Best for: **High-concurrency network I/O** (100,000 concurrent WebSockets/APIs).

---

#### Tricky Follow-up:

* **Question:** *Why is multi-threading ineffective for CPU-bound computations in Python?*

* **Answer:** Because CPython's GIL allows only one thread to execute Python bytecode at a time, causing threads to serialize and context-switch with zero parallel CPU utilization.



In [ ]:
import os
import threading

print("1. Process ID (PID):", os.getpid())
print("2. Active Threads:   ", threading.active_count())
print("3. Concurrency Rule: I/O -> Asyncio/Threading; CPU -> Multiprocessing")


### Q102: How does concurrent.futures (ThreadPoolExecutor vs ProcessPoolExecutor) simplify pool execution?

#### Verbal Answer:

`concurrent.futures` provides a unified high-level async task execution interface:

1. **Core Classes**:
   * `ThreadPoolExecutor(max_workers=...)`: Manages thread workers.
   * `ProcessPoolExecutor(max_workers=...)`: Manages background processes.

2. **Key Methods**:
   * `executor.submit(func, *args)`: Returns a `Future` object immediately.
   * `executor.map(func, iterable)`: Parallel mapping returning ordered results.
   * `as_completed(futures)`: Yields completed futures as soon as they finish.

---

#### Tricky Follow-up:

* **Question:** *What happens if an unhandled exception occurs inside a worker task submitted to a `Future`?*

* **Answer:** The exception is caught and stored in the `Future`. It is re-raised when the caller calls `future.result()`.



In [ ]:
from concurrent.futures import ThreadPoolExecutor

def square(n):
    return n * n

with ThreadPoolExecutor(max_workers=3) as executor:
    results = list(executor.map(square, [1, 2, 3, 4, 5]))

print("ThreadPoolExecutor Parallel Map Output:", results)


### Q103: What are Race Conditions, Deadlocks, and how do threading.Lock and RLock prevent them?

#### Verbal Answer:

Synchronization primitives in multi-threaded programs:

1. **Race Condition**:
   * Multiple threads concurrently read and write shared state, leading to non-deterministic data corruption.

2. **`threading.Lock` (Mutual Exclusion)**:
   * `lock.acquire()` and `lock.release()` (or `with lock:`).
   * Ensures only one thread executes the critical section.

3. **Deadlock**:
   * Two or more threads are permanently blocked, each waiting for a lock held by the other. Prevented by strict hierarchical lock acquisition order.

---

#### Tricky Follow-up:

* **Question:** *How does `lock.acquire(blocking=False)` assist in non-blocking lock polling?*

* **Answer:** It returns `True` immediately if the lock was acquired, or `False` if busy without blocking the calling thread.



In [ ]:
import threading

counter = 0
lock = threading.Lock()

def safe_increment():
    global counter
    with lock:  # Protect critical section
        counter += 1

threads = [threading.Thread(target=safe_increment) for _ in range(50)]
for t in threads: t.start()
for t in threads: t.join()

print("Synchronized Counter with Threading Lock (Target 50):", counter)


### Q104: What is threading.Semaphore and how is it used for Rate-Limiting and Connection Pooling?

#### Verbal Answer:

`threading.Semaphore(value)` manages an internal counter of available resource permits:

1. **Mechanics**:
   * Initialized with `value` (e.g. `value=5` for max 5 concurrent connections).
   * `acquire()`: Decrements counter; blocks if counter == 0.
   * `release()`: Increments counter, waking up blocked threads.

2. **Use Cases**:
   * Rate-limiting API requests to max N concurrent outbound calls.

---

#### Tricky Follow-up:

* **Question:** *What is the difference between a `Semaphore` and a `BoundedSemaphore`?*

* **Answer:** `BoundedSemaphore` raises a `ValueError` if `release()` is called more times than initial `value`, preventing accidental over-releasing bugs.



In [ ]:
import threading
import time

semaphore = threading.BoundedSemaphore(value=2)  # Max 2 concurrent workers

def worker(worker_id):
    with semaphore:
        time.sleep(0.01)
        return f"Worker {worker_id} completed."

threads = [threading.Thread(target=worker, args=(i,)) for i in range(4)]
for t in threads: t.start()
for t in threads: t.join()

print("BoundedSemaphore successfully limited concurrent workers.")


### Q105: How do threading.Event and threading.Condition enable inter-thread signaling and coordination?

#### Verbal Answer:

Inter-thread communication primitives:

1. **`threading.Event`**:
   * A thread-safe boolean flag.
   * `event.set()`: Sets flag to True and wakes all waiting threads.
   * `event.wait()`: Blocks thread until flag is True.
   * `event.clear()`: Resets flag to False.

2. **`threading.Condition`**:
   * Coordinates producer-consumer relationships with `cond.wait()` and `cond.notify()`.

---

#### Tricky Follow-up:

* **Question:** *Why is `threading.Condition` preferred over polling loops with `time.sleep()`?*

* **Answer:** `Condition.wait()` puts the thread to sleep in the OS scheduler with zero CPU consumption until `notify()` wakes it directly.



In [ ]:
import threading

ready_event = threading.Event()

def service_worker():
    ready_event.wait()  # Waits for startup signal
    return "Service started"

t = threading.Thread(target=service_worker)
t.start()

# Signal event
ready_event.set()
t.join()

print("threading.Event signaling successfully coordinated thread execution.")


### Q106: How do multiprocessing.Queue, Pipe, and Value/Array share data and state across processes?

#### Verbal Answer:

Inter-Process Communication (IPC) mechanisms:

1. **`multiprocessing.Queue`**:
   * Thread and process-safe FIFO queue backed by OS pipes and background feeder threads (data is pickled/unpickled).

2. **`multiprocessing.Pipe(duplex=True)`**:
   * Fast two-way point-to-point communication channel between 2 processes.

3. **`multiprocessing.Value` and `Array` (Shared Memory)**:
   * Allocated in C-level shared memory (no pickling overhead); guarded by mutex locks.

---

#### Tricky Follow-up:

* **Question:** *What is the fastest way to share massive numeric matrices across Python processes?*

* **Answer:** Use `multiprocessing.shared_memory.SharedMemory` (Python 3.8+) or NumPy `np.memmap` for zero-copy memory sharing.



In [ ]:
import multiprocessing
from multiprocessing import Value, Array

# Shared memory scalar and array (C-level)
shared_num = Value('d', 0.0)      # double
shared_arr = Array('i', range(5)) # array of ints

print("1. Shared memory scalar initial:   ", shared_num.value)
print("2. Shared memory array first value:", shared_arr[0])


### Q107: What are the differences between spawn, fork, and forkserver process start methods (multiprocessing.set_start_method)?

#### Verbal Answer:

Process Start Methods (PEP 596):

1. **`spawn` (Default on Windows and macOS)**:
   * Starts a brand new clean Python interpreter process.
   * Safest method; avoids inheriting locks or open file descriptors.
   * Slower startup; requires pickling all objects passed to worker.

2. **`fork` (Legacy Linux Default)**:
   * Copies memory pages via OS copy-on-write. Fast startup.
   * **Hazard**: Inherits deadlocks if another thread was holding a lock during `fork` (deprecated in Python 3.12+ macOS/Linux).

3. **`forkserver`**:
   * Spawns a clean single-threaded server process from which subsequent workers are forked safely.

---

#### Tricky Follow-up:

* **Question:** *Why is `if __name__ == '__main__':` strictly mandatory when using multiprocessing with `spawn`?*

* **Answer:** Because `spawn` re-imports the main module in each child process. Without the guard, it triggers an infinite recursive loop of child processes spawning children.



In [ ]:
import multiprocessing

start_method = multiprocessing.get_start_method()
print("Active Multiprocessing Process Start Method:", start_method)


### Q108: What is multiprocessing.shared_memory (SharedMemory, ShareableList) introduced in Python 3.8?

#### Verbal Answer:

`multiprocessing.shared_memory` provides **Zero-Copy Inter-Process Shared Memory**:

1. **Mechanism**:
   * Allocates a named block of POSIX/Windows shared memory.
   * Multiple independent processes attach to the same memory buffer by name without serializing or copying bytes over pipes.

2. **NumPy Integration**:
   * Wrap raw shared memory directly in a NumPy array: `np.ndarray(shape, dtype, buffer=shm.buf)`.

---

#### Tricky Follow-up:

* **Question:** *What happens if a process creates a `SharedMemory` block and forgets to call `shm.unlink()`?*

* **Answer:** The shared memory block persists in operating system RAM even after the Python process terminates, causing an OS-level memory leak.



In [ ]:
from multiprocessing import shared_memory
import numpy as np

# Allocate 1000 float64s in shared memory
shm = shared_memory.SharedMemory(create=True, size=8000)
shared_np = np.ndarray((1000,), dtype=np.float64, buffer=shm.buf)
shared_np[:] = 42.0

print("1. Shared Memory Block Name:", shm.name)
print("2. Shared NumPy Value[0]:   ", shared_np[0])

# Clean up
shm.close()
shm.unlink()


### Q109: How does ThreadLocal (threading.local) isolate thread-specific context and state?

#### Verbal Answer:

`threading.local()` stores attributes specific to the currently executing thread:

1. **Mechanism**:
   * Looking up `thread_local.user_id` returns the value set by the **current thread**, completely invisible and isolated from all other threads.

2. **Use Cases**:
   * Storing database transaction handles or HTTP request context in multi-threaded web servers (Flask/Django).

---

#### Tricky Follow-up:

* **Question:** *What is the asynchronous equivalent to `threading.local()` in modern asyncio?*

* **Answer:** Use `contextvars.ContextVar` (PEP 567).



In [ ]:
import threading

local_ctx = threading.local()

def thread_task(name):
    local_ctx.user = name
    print(f"[{threading.current_thread().name}] Local User: {local_ctx.user}")

t1 = threading.Thread(target=thread_task, args=('Alice',), name="Thread-A")
t2 = threading.Thread(target=thread_task, args=('Bob',), name="Thread-B")

t1.start(); t2.start()
t1.join(); t2.join()


### Q110: What are Daemon Threads in Python and what are the risks of using them?

#### Verbal Answer:

Daemon Threads (`thread.daemon = True`):

1. **Behavior**:
   * Background support threads (e.g. heartbeat pings, garbage sweepers).
   * Python process **abruptly terminates immediately** when all non-daemon threads finish, killing daemon threads instantly in the middle of execution.

2. **Production Risk**:
   * Daemon threads are killed without executing `finally:` blocks or context manager cleanup, risking database corruption and incomplete file writes. Use graceful shutdown events instead.

---

#### Tricky Follow-up:

* **Question:** *How do you ensure a thread is NOT a daemon thread?*

* **Answer:** Set `thread.daemon = False` (default for standard threads).



In [ ]:
import threading

daemon_t = threading.Thread(target=lambda: None, daemon=True)
standard_t = threading.Thread(target=lambda: None, daemon=False)

print("1. Is daemon_t daemon?", daemon_t.daemon)
print("2. Is standard_t daemon?", standard_t.daemon)


## Category 12: Asynchronous Programming (asyncio)


### Q111: How does the asyncio Event Loop, Coroutines, and Cooperative Multitasking operate under the hood?

#### Verbal Answer:

Asyncio & Cooperative Multitasking Architecture:

1. **Event Loop**:
   * Single-threaded loop backed by OS I/O multiplexing (`epoll` on Linux, `kqueue` on macOS, `IOCP` on Windows).
   * Polls sockets and timers, dispatching execution to ready coroutines.

2. **Cooperative Scheduling**:
   * Coroutines voluntarily yield control back to the event loop upon hitting an `await` expression.
   * Zero OS thread switching overhead; can handle 50,000+ concurrent connections in a single thread.

---

#### Tricky Follow-up:

* **Question:** *What happens if a coroutine executes a long-running CPU loop or synchronous `time.sleep()` without `await`?*

* **Answer:** It freezes the entire event loop, blocking all other concurrent tasks from progressing.



In [ ]:
import asyncio

async def fetch_market_tick(source, delay):
    await asyncio.sleep(delay)  # Yields control back to event loop
    return f"Tick from {source}"

async def main():
    res1 = await fetch_market_tick("NYSE", 0.01)
    res2 = await fetch_market_tick("NASDAQ", 0.01)
    print("Asyncio execution results:", res1, res2)

asyncio.run(main())


### Q112: What is the difference between asyncio.gather, asyncio.wait, and asyncio.TaskGroup (Python 3.11+)?

#### Verbal Answer:

Concurrent Task Aggregators:

1. **`asyncio.gather(*coros)`**:
   * Runs coroutines concurrently and returns list of results in original order.
   * `return_exceptions=True` captures exceptions as result elements.

2. **`asyncio.wait(tasks, return_when=...)`**:
   * Returns `(done_set, pending_set)` upon completion of `FIRST_COMPLETED` or `ALL_COMPLETED`.

3. **`asyncio.TaskGroup()` (Modern Best Practice - Structured Concurrency)**:
   * Context manager `async with asyncio.TaskGroup() as tg:`.
   * If any task fails, cancels all other running tasks automatically and groups errors into an `ExceptionGroup`.

---

#### Tricky Follow-up:

* **Question:** *Why is `asyncio.TaskGroup` superior to `asyncio.gather` for error handling?*

* **Answer:** `TaskGroup` prevents orphaned/leaked background tasks by automatically cancelling sibling tasks when one fails.



In [ ]:
import asyncio

async def worker(task_id):
    await asyncio.sleep(0.01)
    return f"Task_{task_id}_Done"

async def main():
    # 1. asyncio.gather
    results = await asyncio.gather(worker(1), worker(2))
    print("1. asyncio.gather results:", results)

    # 2. Modern asyncio.TaskGroup (Python 3.11+)
    async with asyncio.TaskGroup() as tg:
        t1 = tg.create_task(worker(3))
        t2 = tg.create_task(worker(4))
    print("2. TaskGroup results:    ", [t1.result(), t2.result()])

asyncio.run(main())


### Q113: How does asyncio.to_thread / run_in_executor offload blocking synchronous I/O or CPU code?

#### Verbal Answer:

Bridging synchronous code into Asyncio:

1. **`asyncio.to_thread(sync_func, *args)` (Python 3.9+)**:
   * Offloads blocking synchronous code (e.g. `requests.get()` or legacy database drivers) to a background thread pool without blocking the event loop.

2. **`loop.run_in_executor(executor, sync_func)`**:
   * Offloads to a custom `ThreadPoolExecutor` or `ProcessPoolExecutor` for CPU-heavy tasks.

---

#### Tricky Follow-up:

* **Question:** *Can `asyncio.to_thread` be used for CPU-heavy tasks to achieve multi-core parallel speedups?*

* **Answer:** No, because background threads in `to_thread` still share the GIL. CPU tasks require `ProcessPoolExecutor`.



In [ ]:
import asyncio
import time

def blocking_io_operation(x):
    time.sleep(0.01)  # Synchronous blocking call
    return x * 10

async def main():
    # Offload to background thread safely
    result = await asyncio.to_thread(blocking_io_operation, 5)
    print("Non-blocking result from asyncio.to_thread:", result)

asyncio.run(main())


### Q114: What are Async Iterators (async for, __aiter__, __anext__) and Async Generators (async def with yield)?

#### Verbal Answer:

Asynchronous Streaming Protocols:

1. **Async Generator**:
   * `async def my_generator(): await sleep(); yield val`
   * Combines coroutines with generator streaming.

2. **`async for` Loop**:
   * Consumes async generators as values arrive over network sockets.

---

#### Tricky Follow-up:

* **Question:** *What exception terminates an async iterator?*

* **Answer:** `StopAsyncIteration` (raised by `__anext__()`).



In [ ]:
import asyncio

async def async_event_stream():
    for i in range(3):
        await asyncio.sleep(0.01)
        yield f"Event_{i}"

async def main():
    results = []
    async for event in async_event_stream():
        results.append(event)
    print("Async Generator Stream:", results)

asyncio.run(main())


### Q115: How do you enforce Timeouts and handle Task Cancellation in asyncio (asyncio.timeout, wait_for)?

#### Verbal Answer:

Timeout and Cancellation Management:

1. **`asyncio.timeout(delay)` (Python 3.11+ Context Manager)**:
   * `async with asyncio.timeout(2.0): await task`
   * Raises `TimeoutError` if block exceeds deadline.

2. **Task Cancellation**:
   * Calling `task.cancel()` injects `asyncio.CancelledError` into the awaiting coroutine.
   * Cleanup can be performed in `except asyncio.CancelledError:` or `finally: ...`.

---

#### Tricky Follow-up:

* **Question:** *What must you do if you catch `asyncio.CancelledError` in a coroutine?*

* **Answer:** Perform cleanup and re-raise `CancelledError` (or allow it to propagate) so the task cancellation completes.



In [ ]:
import asyncio

async def long_running_task():
    try:
        await asyncio.sleep(10)
    except asyncio.CancelledError:
        return "Cleanup performed on CancelledError"

async def main():
    task = asyncio.create_task(long_running_task())
    await asyncio.sleep(0.01)
    task.cancel()
    res = await task
    print("Task Cancellation Handled:", res)

asyncio.run(main())


### Q116: How does contextvars.ContextVar maintain thread-safe and async-safe context (request IDs, trace IDs)?

#### Verbal Answer:

`contextvars` (PEP 567) provides **Async-Safe Context Isolation**:

1. **The Problem with ThreadLocal in Asyncio**:
   * In asyncio, thousands of concurrent tasks execute on the **same single OS thread**, causing `threading.local` state to bleed across tasks.

2. **ContextVar Solution**:
   * Isolates variables per **asynchronous task execution context**.
   * Essential for storing distributed OpenTelemetry trace IDs and HTTP request headers in FastAPI / Starlette.

---

#### Tricky Follow-up:

* **Question:** *What happens when a new async task is created via `asyncio.create_task()`?*

* **Answer:** The new task inherits a shallow copy of the parent task's current `contextvars` context.



In [ ]:
import asyncio
import contextvars

request_id_var = contextvars.ContextVar('request_id', default="NO_REQ")

async def log_with_req_id(msg):
    return f"[{request_id_var.get()}] {msg}"

async def handle_request(req_id):
    request_id_var.set(req_id)
    return await log_with_req_id("Processing order")

async def main():
    r1, r2 = await asyncio.gather(
        handle_request("REQ_1001"),
        handle_request("REQ_1002")
    )
    print("ContextVar Isolation:", r1, r2)

asyncio.run(main())


### Q117: What is asyncio.Queue and how does it implement Asynchronous Producer-Consumer pipelines?

#### Verbal Answer:

`asyncio.Queue` provides non-blocking FIFO queues for asynchronous coroutines:

1. **Key Methods**:
   * `await queue.put(item)`: Suspends if queue is full (`maxsize`).
   * `await queue.get()`: Suspends until an item is available.
   * `queue.task_done()`: Decrements unfinished task counter.
   * `await queue.join()`: Blocks until all items are processed.

---

#### Tricky Follow-up:

* **Question:** *How do you signal consumer coroutines to shut down cleanly in an async queue?*

* **Answer:** Put a sentinel value (like `None`) into the queue for each active worker.



In [ ]:
import asyncio

async def main():
    q = asyncio.Queue()
    await q.put("Event_A")
    await q.put("Event_B")

    item1 = await q.get()
    q.task_done()
    item2 = await q.get()
    q.task_done()

    print("Async Queue Processed Items:", item1, item2)

asyncio.run(main())


### Q118: What is asyncio.shield and how does it protect critical tasks from cancellation?

#### Verbal Answer:

`asyncio.shield(task)` prevents cancellation from propagating down to an inner task:

1. **Use Case**:
   * Critical operations (e.g. committing a database transaction or logging an audit trail) that must complete even if the outer HTTP request was aborted/cancelled by the client.

---

#### Tricky Follow-up:

* **Question:** *What happens if the outer task is cancelled while awaiting `asyncio.shield(inner_task)`?*

* **Answer:** The outer awaiter raises `CancelledError`, but the `inner_task` continues running to completion in the background.



In [ ]:
import asyncio

async def critical_audit_log():
    await asyncio.sleep(0.01)
    return "Audit Logged Successfully"

async def main():
    task = asyncio.create_task(critical_audit_log())
    # Shield protects task from outer cancellations
    res = await asyncio.shield(task)
    print("Shielded task completed:", res)

asyncio.run(main())


### Q119: What is the difference between asyncio.Future and asyncio.Task?

#### Verbal Answer:

Futures vs Tasks in Asyncio:

1. **`asyncio.Future`**:
   * Low-level representation of an eventual result (callback placeholder).
   * Has no execution logic; someone must manually call `future.set_result(val)` or `future.set_exception(e)`.

2. **`asyncio.Task`**:
   * Subclass of `Future` that **wraps and drives a coroutine**.
   * Automatically scheduled on the event loop by `asyncio.create_task()` and sets its own result when the coroutine finishes.

---

#### Tricky Follow-up:

* **Question:** *How can you check if a task is finished without awaiting it?*

* **Answer:** Call `task.done()`.



In [ ]:
import asyncio

async def sample_coro():
    return 42

async def main():
    task = asyncio.create_task(sample_coro())
    print("1. Is Task instance of Future?", isinstance(task, asyncio.Future))
    res = await task
    print("2. Task Result:", res)

asyncio.run(main())


### Q120: How do you build an Asynchronous Rate-Limiter using asyncio.Semaphore?

#### Verbal Answer:

Asynchronous Rate Limiting:

1. **Pattern**:
   * `sem = asyncio.Semaphore(max_concurrent_requests)`
   * Wrap async outbound HTTP requests in `async with sem:`.
   * Guarantees at most N concurrent sockets are open simultaneously, protecting against downstream API throttling (HTTP 429 Too Many Requests).

---

#### Tricky Follow-up:

* **Question:** *How can you combine an `asyncio.Semaphore` with a time window (e.g. max 100 requests per second)?*

* **Answer:** Use an async token bucket algorithm using `asyncio.sleep()` or libraries like `aiolimiter`.



In [ ]:
import asyncio

sem = asyncio.Semaphore(2)  # Max 2 concurrent requests

async def mock_api_fetch(req_id):
    async with sem:
        await asyncio.sleep(0.01)
        return f"Req_{req_id}_OK"

async def main():
    results = await asyncio.gather(*(mock_api_fetch(i) for i in range(4)))
    print("Rate-Limited Async Requests Completed:", results)

asyncio.run(main())


## Category 13: Metaprogramming, Descriptors & Metaclasses


### Q121: What is the Descriptor Protocol (__get__, __set__, __delete__) and how does it power @property, methods, and ORMs?

#### Verbal Answer:

The Descriptor Protocol is the foundation of Python's Object-Oriented machinery:

1. **Protocol Methods**:
   * `__get__(self, instance, owner=None)`: Called on attribute retrieval.
   * `__set__(self, instance, value)`: Called on attribute assignment.
   * `__delete__(self, instance)`: Called on `del obj.attr`.
   * `__set_name__(self, owner, name)` (Python 3.6+): Automatically captures the attribute name.

2. **Everyday Built-in Descriptors**:
   * `@property`, `@classmethod`, `@staticmethod`, and regular functions (which bind `self` via `__get__`) are all descriptors!
   * Power Django/SQLAlchemy ORM field validation (`models.CharField()`).

---

#### Tricky Follow-up:

* **Question:** *What is the difference between a Data Descriptor and a Non-Data Descriptor?*

* **Answer:** A Data Descriptor defines `__set__` or `__delete__` and takes precedence over instance `__dict__`; a Non-Data Descriptor defines only `__get__` and can be shadowed by instance `__dict__`.



In [ ]:
class NonNegative:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None: return self
        return instance.__dict__.get(self.name, 0)

    def __set__(self, instance, value):
        if value < 0:
            raise ValueError(f"{self.name} cannot be negative!")
        instance.__dict__[self.name] = value

class Product:
    price = NonNegative()

p = Product()
p.price = 100
print("Descriptor validated price attribute:", p.price)


### Q122: What is __set_name__ introduced in Python 3.6 (PEP 487) and how does it eliminate descriptor boilerplate?

#### Verbal Answer:

`__set_name__(self, owner, name)` automates attribute name capture:

1. **Prior to Python 3.6**:
   * Descriptors had to be initialized with redundant name arguments: `price = Field('price')`.

2. **PEP 487 Automation**:
   * When the class is constructed, Python automatically calls `descriptor.__set_name__(owner_class, attribute_name)`.
   * Completely eliminates redundant name passing in ORMs and schema validators.

---

#### Tricky Follow-up:

* **Question:** *When exactly is `__set_name__` called during class creation?*

* **Answer:** It is called automatically by `type.__new__` after the class body is evaluated and the class object is instantiated.



In [ ]:
class AutoNameDescriptor:
    def __set_name__(self, owner, name):
        self.field_name = name

    def __get__(self, instance, owner):
        return f"Bound to attribute name: '{self.field_name}'"

class OrderModel:
    order_id = AutoNameDescriptor()
    customer_name = AutoNameDescriptor()

o = OrderModel()
print("1.", o.order_id)
print("2.", o.customer_name)


### Q123: What is __init_subclass__ (PEP 487) and why is it preferred over Metaclasses for subclass customization?

#### Verbal Answer:

`__init_subclass__` provides clean subclass hook customization without metaclasses:

1. **Mechanism**:
   * Defined as a class method on a parent class: `def __init_subclass__(cls, **kwargs):`.
   * Automatically executes **whenever a subclass is defined**, receiving the new subclass `cls`.

2. **Advantages over Metaclasses**:
   * Zero metaclass conflicts in multiple inheritance.
   * Far simpler syntax for plugin registration, schema enforcement, and auto-validation.

---

#### Tricky Follow-up:

* **Question:** *Can `__init_subclass__` accept keyword arguments passed in the class definition header?*

* **Answer:** Yes: `class PluginA(BasePlugin, version='1.0'):` passes `version='1.0'` directly to `__init_subclass__`.



In [ ]:
class PluginRegistryBase:
    registry = {}

    def __init_subclass__(cls, plugin_name=None, **kwargs):
        super().__init_subclass__(**kwargs)
        name = plugin_name or cls.__name__
        cls.registry[name] = cls

class PDFExporter(PluginRegistryBase, plugin_name="pdf_v1"): pass
class CSVExporter(PluginRegistryBase, plugin_name="csv_v1"): pass

print("Auto-Registered Plugins via __init_subclass__:", PluginRegistryBase.registry)


### Q124: What is a Metaclass in Python and how do you customize class creation using type.__new__?

#### Verbal Answer:

A **Metaclass** is the 'class of a class' (its instances are classes):

1. **The Construction Hierarchy**:
   * `instance` is an instance of `Class`.
   * `Class` is an instance of `type` (or a custom metaclass inheriting from `type`).

2. **Lifecycle Methods**:
   * `__prepare__(name, bases)`: Returns the namespace mapping (dict) used during class body evaluation.
   * `__new__(mcs, name, bases, namespace)`: Allocates the physical class object in memory.
   * `__init__(cls, name, bases, namespace)`: Initializes the created class object.

---

#### Tricky Follow-up:

* **Question:** *What is the Metaclass Conflict error in Python?*

* **Answer:** Occurs when a class inherits from multiple parents whose metaclasses are not subtypes of one another. Resolved by defining a composite derived metaclass.



In [ ]:
class EnforceUppercaseMeta(type):
    def __new__(mcs, name, bases, dct):
        # Enforce all methods are uppercase
        uppercase_attrs = {}
        for k, v in dct.items():
            if not k.startswith("__") and callable(v):
                uppercase_attrs[k.upper()] = v
            else:
                uppercase_attrs[k] = v
        return super().__new__(mcs, name, bases, uppercase_attrs)

class Service(metaclass=EnforceUppercaseMeta):
    def execute_job(self): return "Job finished"

s = Service()
print("Metaclass modified method name to uppercase:", hasattr(s, 'EXECUTE_JOB'))


### Q125: What is the __prepare__ method on a Metaclass and why is it used to track class attribute declaration order?

#### Verbal Answer:

`__prepare__(name, bases, **kwargs)` creates the namespace dict for the class body:

1. **How It Works**:
   * Called **before** the class body is executed.
   * Returning a custom dictionary mapping (like `collections.OrderedDict` in older Python, or a custom validation dict) allows intercepting attribute definitions as they are written in the class body.

2. **Use Cases**:
   * Building enum classes and ORMs where duplicate attribute names must be rejected immediately upon definition.

---

#### Tricky Follow-up:

* **Question:** *Must `__prepare__` be declared as a `@classmethod` or standard method on the metaclass?*

* **Answer:** It can be declared with `@classmethod` or as a regular method on the metaclass (Python automatically treats it as a class method).



In [ ]:
class OrderedMeta(type):
    @classmethod
    def __prepare__(mcs, name, bases):
        return {}  # Can return custom dict to track attribute definitions

class ExampleClass(metaclass=OrderedMeta):
    a = 1
    b = 2

print("Class created cleanly with __prepare__ hook:", ExampleClass.a, ExampleClass.b)


### Q126: How does Monkey Patching work in Python, and what are its legitimate testing uses vs production hazards?

#### Verbal Answer:

Monkey Patching is the dynamic modification of a module or class at runtime:

1. **Mechanism**:
   * `target_module.api_call = mock_function`

2. **Legitimate Uses**:
   * Unit testing (mocking network calls, database responses).
   * Critical hot-patching of third-party bugs before an upstream patch is released.

3. **Production Hazards**:
   * Obscures code paths, creates spooky action-at-a-distance bugs, and breaks static type checkers.

---

#### Tricky Follow-up:

* **Question:** *What tool in the Python standard library provides safe, scoped monkey patching for unit tests?*

* **Answer:** `unittest.mock.patch` (which automatically restores original attributes upon exit).



In [ ]:
class RemoteService:
    def fetch(self):
        return "Live Network Data"

# Monkey patch for testing
def mock_fetch(self):
    return "Mocked Offline Data"

original_fetch = RemoteService.fetch
RemoteService.fetch = mock_fetch

service = RemoteService()
print("Monkey patched execution result:", service.fetch())

# Restore
RemoteService.fetch = original_fetch


### Q127: How do type(name, bases, dict) dynamically construct classes at runtime?

#### Verbal Answer:

3-Argument `type()` Dynamic Class Creation:

`DynamicClass = type('ClassName', (BaseClass,), {'attribute': value, 'method': func})`

1. **Parameters**:
   * `name`: String name of class.
   * `bases`: Tuple of parent classes.
   * `dict`: Dictionary of class attributes and methods.

2. **Use Case**:
   * Dynamic ORM models, code generators, and deserializers constructing classes from JSON schemas.

---

#### Tricky Follow-up:

* **Question:** *What is the difference between `type(obj)` with 1 argument vs 3 arguments?*

* **Answer:** 1 argument returns the type of `obj`; 3 arguments dynamically constructs a brand new class.



In [ ]:
# Dynamic class creation with 3-argument type()
def greet(self):
    return f"Hello from dynamic class instance {self.name}"

DynamicUser = type('DynamicUser', (object,), {
    'name': 'Default User',
    'greet': greet
})

user_inst = DynamicUser()
print("Dynamically created class instance:", user_inst.greet())


### Q128: What is __class_getitem__ (PEP 560) and how does it support Generic Type Subscripting (MyClass[int])?

#### Verbal Answer:

`__class_getitem__` enables generic subscripting syntax `Class[Type]` on custom classes:

1. **How It Works**:
   * Class method on a class (called when indexing the class itself: `MyClass[T]`).
   * Standardizes generic type annotations without requiring custom metaclasses.

---

#### Tricky Follow-up:

* **Question:** *What is the difference between `__getitem__` and `__class_getitem__`?*

* **Answer:** `__getitem__` is called when subscripting an **instance** `obj[key]`; `__class_getitem__` is called when subscripting the **class itself** `Class[key]`.



In [ ]:
class CustomContainer:
    def __class_getitem__(cls, item):
        return f"Generic Container specialized for type: {item}"

print("Generic Subscripting Result (CustomContainer[int]):", CustomContainer[int])


### Q129: How does sys.settrace enable Debuggers (pdb), Profilers (cProfile), and Code Coverage tools (coverage.py)?

#### Verbal Answer:

`sys.settrace(trace_func)` is Python's execution tracing engine:

1. **Trace Function Events**:
   * Intercepts 5 core interpreter events:
     * `'call'`: Function entered.
     * `'line'`: New line about to execute.
     * `'return'`: Function about to return.
     * `'exception'`: Exception raised.
     * `'opcode'`: Bytecode step.

2. **Core Applications**:
   * Setting breakpoints (`pdb`), recording line coverage (`coverage.py`), and step debugging.

---

#### Tricky Follow-up:

* **Question:** *Why does running code under `sys.settrace` slow down execution by 10x to 50x?*

* **Answer:** Because CPython disables fast VM loops and invokes the Python trace callback function on every single executed line.



In [ ]:
import sys

lines_executed = []
def simple_tracer(frame, event, arg):
    if event == 'line':
        lines_executed.append(frame.f_lineno)
    return simple_tracer

# Trace a small block
sys.settrace(simple_tracer)
x = 10
y = 20
z = x + y
sys.settrace(None)

print("Line numbers traced by sys.settrace:", len(lines_executed) > 0)


### Q130: What are Code Objects (PyCodeObject) and how does types.CodeType allow compiling dynamic bytecode?

#### Verbal Answer:

`PyCodeObject` is the immutable compiled unit of Python executable logic:

1. **Key Attributes**:
   * `co_code`: Raw bytecode instruction bytes.
   * `co_consts`: Tuple of literal constants.
   * `co_varnames`: Tuple of local variable names.
   * `co_names`: Global names referenced.

2. **`compile(source, filename, mode)`**:
   * Returns a `code` object that can be passed to `exec()` or `eval()`.

---

#### Tricky Follow-up:

* **Question:** *Why is `compile()` + `exec()` faster than calling `exec(string)` repeatedly in a loop?*

* **Answer:** Pre-compiling with `compile()` parses the AST and generates bytecode once, eliminating compilation overhead on each iteration.



In [ ]:
code_obj = compile("result = a * 10 + b", "<dynamic>", "exec")

loc = {'a': 5, 'b': 2}
exec(code_obj, {}, loc)

print("Pre-compiled Code Object Execution Result:", loc['result'])


## Category 14: CPython Internals, GC & Low-Level Memory Layout


### Q131: What is PyObject in CPython and what are the exact memory overheads of integers, floats, and strings in 64-bit Python?

#### Verbal Answer:

Memory Overheads in 64-bit CPython:

1. **`PyObject` Header**:
   * 8 bytes `ob_refcnt` + 8 bytes `ob_type` = **16 bytes baseline**.

2. **Type Overheads**:
   * **`int` (`PyLongObject`)**: 28 bytes minimum (16-byte header + 8-byte size + 4-byte digit array).
   * **`float` (`PyFloatObject`)**: 24 bytes (16-byte header + 8-byte C double).
   * **`str` (`PyASCIIObject`)**: 49–50 bytes minimum for empty string.
   * **`list` (`PyListObject`)**: 56 bytes + 8 bytes per item pointer.
   * **`dict` (`PyDictObject`)**: 64–128 bytes baseline.

---

#### Tricky Follow-up:

* **Question:** *Why does a 1,000,000 integer NumPy array consume only ~8MB while a Python list of 1,000,000 integers consumes ~36MB?*

* **Answer:** NumPy stores raw 8-byte integers contiguously; Python list stores 8MB of pointers + 28MB of separate heap `PyLongObject` instances.



In [ ]:
import sys

print("1. Integer 42 size:  ", sys.getsizeof(42), "bytes")
print("2. Float 42.0 size:   ", sys.getsizeof(42.0), "bytes")
print("3. Empty string size: ", sys.getsizeof(""), "bytes")
print("4. Empty list size:   ", sys.getsizeof([]), "bytes")
print("5. Empty dict size:   ", sys.getsizeof({}), "bytes")


### Q132: How does PyMalloc (Small Object Allocator) manage Arenas, Pools, and Blocks in CPython?

#### Verbal Answer:

`PyMalloc` is CPython's specialized memory allocator for small objects (<= 512 bytes):

1. **Hierarchy**:
   * **Arenas (256 KB)**: Allocated from OS via `malloc()`.
   * **Pools (4 KB, matching OS page size)**: Contained inside Arenas. Each Pool handles a specific size class (e.g. 16-byte blocks, 32-byte blocks).
   * **Blocks (8 to 512 bytes)**: Exact chunk carved for individual `PyObject`s.

2. **Why It Exists**:
   * Avoids severe heap fragmentation and system call overhead from millions of tiny `malloc()` calls.

---

#### Tricky Follow-up:

* **Question:** *Why does Python sometimes not release memory back to the OS after `del` on millions of objects?*

* **Answer:** Because an entire 256KB Arena must be 100% empty of all living objects before `PyMalloc` can free it back to the OS kernel.



In [ ]:
print("PyMalloc Architecture: 256KB Arenas -> 4KB Pools -> 8-512B Blocks")


### Q133: How do you tune and debug Garbage Collection thresholds using the gc module (gc.get_threshold, set_threshold)?

#### Verbal Answer:

Tuning Generational GC with the `gc` module:

1. **Generations & Thresholds**:
   * `gc.get_threshold()` returns `(threshold0, threshold1, threshold2)` (default `(700, 10, 10)`).
   * `threshold0`: Number of net allocations over deallocations triggering a Gen 0 collection.

2. **Performance Tuning**:
   * Increasing thresholds (`gc.set_threshold(50000, 20, 20)`) reduces GC pause latency in high-throughput real-time pipelines.
   * `gc.disable()` temporarily disables cyclic GC during time-critical trading execution loops.

---

#### Tricky Follow-up:

* **Question:** *How can you find what objects are keeping an orphaned object alive?*

* **Answer:** Use `gc.get_referrers(target_obj)` to inspect all incoming pointers.



In [ ]:
import gc

print("1. Default GC Thresholds (Gen 0, 1, 2):", gc.get_threshold())
print("2. GC Count per generation:           ", gc.get_count())


### Q134: What is Python 3.11's Specializing Adaptive Interpreter (PEP 659) and Quickening?

#### Verbal Answer:

Python 3.11+ Faster CPython Optimization Engine:

1. **Quickening & Specialization**:
   * Identifies frequently executed 'hot' bytecode instructions and replaces generic opcodes with type-specialized opcodes:
     * `BINARY_OP` specialized into `BINARY_OP_ADD_INT` or `BINARY_OP_ADD_FLOAT`.
     * `LOAD_ATTR` specialized into `LOAD_ATTR_INSTANCE_VALUE`.

2. **Result**:
   * Up to **10-60% speedups** in pure Python code without changing a single line of code!

---

#### Tricky Follow-up:

* **Question:** *What happens if a specialized opcode encounters an argument of an unexpected type at runtime?*

* **Answer:** It 'de-optimizes' back to the generic opcode without raising an error.



In [ ]:
import dis

def add_integers(a, b):
    return a + b

# Warm up JIT adaptive interpreter
for _ in range(1000):
    add_integers(10, 20)

print("Adaptive Specialization active on bytecode instructions.")


### Q135: What is memoryview and how does the Buffer Protocol enable zero-copy byte slicing?

#### Verbal Answer:

The Buffer Protocol & `memoryview`:

1. **Buffer Protocol (C-level)**:
   * Allows Python objects (`bytes`, `bytearray`, `array.array`, NumPy arrays) to expose raw memory pointer buffers.

2. **`memoryview` (Zero-Copy Slicing)**:
   * Slicing a `bytes` object `b[0:1000]` creates a **copy** of 1,000 bytes.
   * Slicing a `memoryview(b)[0:1000]` creates a **zero-copy view** referencing the original memory buffer!
   * Critical for high-throughput socket servers (handling gigabytes/sec without RAM thrashing).

---

#### Tricky Follow-up:

* **Question:** *Can a `memoryview` mutate underlying memory in-place?*

* **Answer:** Yes, if created over a mutable buffer like `bytearray` or NumPy array.



In [ ]:
# Zero-copy memoryview mutation over bytearray
buffer = bytearray(b"abcdefgh")
mv = memoryview(buffer)

# Slice without copying
sub_view = mv[2:5]
sub_view[0] = ord('X')  # Mutates buffer in-place!

print("Mutated Underlying Buffer via memoryview:", buffer)


### Q136: How do you detect memory leaks in Python using tracemalloc and objgraph?

#### Verbal Answer:

Memory Leak Profiling with `tracemalloc`:

1. **`tracemalloc` Module**:
   * Tracks exact Python memory allocations back to file names and line numbers.

2. **Snapshot Comparison**:
   * `snapshot1 = tracemalloc.take_snapshot()`
   * Execute workload.
   * `snapshot2 = tracemalloc.take_snapshot()`
   * `top_stats = snapshot2.compare_to(snapshot1, 'lineno')` identifies the exact line leaking RAM.

---

#### Tricky Follow-up:

* **Question:** *What is the primary cause of memory leaks in Python given that it has garbage collection?*

* **Answer:** Unbounded caches (growing dictionaries/lists), active global event listeners, and uncollected circular references in long-running processes.



In [ ]:
import tracemalloc

tracemalloc.start()
snap1 = tracemalloc.take_snapshot()

# Simulated allocation
leak_list = [x for x in range(10000)]

snap2 = tracemalloc.take_snapshot()
top_diff = snap2.compare_to(snap1, 'lineno')

print("Top Memory Allocation Delta Line:")
for stat in top_diff[:1]:
    print(" ", stat)
tracemalloc.stop()


### Q137: What is the __builtins__ namespace dictionary and how does Python look up built-in symbols?

#### Verbal Answer:

The `__builtins__` module namespace:

1. **Mechanism**:
   * Top-level built-in dictionary (`builtins.__dict__`) holding functions like `print`, `len`, `range`.
   * Final fallback layer in the LEGB scope resolution rule.

2. **Security Gotcha**:
   * Sandboxes attempting to secure Python by deleting `__builtins__['eval']` can be bypassed via object traversal (`().__class__.__base__.__subclasses__()`).

---

#### Tricky Follow-up:

* **Question:** *Why is Python sandboxing via restricted `__builtins__` considered insecure?*

* **Answer:** Because Python's dynamic object model allows navigating from any object up to `object.__subclasses__()` to recover `os.system` and file descriptors.



In [ ]:
import builtins

print("1. 'len' exists in builtins dict:", 'len' in builtins.__dict__)
print("2. Accessing built-in directly:   ", builtins.abs(-42))


### Q138: How does ctypes allow calling C shared libraries (.so / .dll) and mapping C-structs directly from Python?

#### Verbal Answer:

`ctypes` foreign function interface (FFI):

1. **Loading C Libraries**:
   * `libc = ctypes.CDLL("libc.so.6")` (Unix) or `ctypes.cdll.msvcrt` (Windows).

2. **Defining C Structures**:
   * `class Point(ctypes.Structure): _fields_ = [('x', ctypes.c_int), ('y', ctypes.c_int)]`

3. **Function Signatures**:
   * Set `func.argtypes = [...]` and `func.restype = ...` for C-level type safety.

---

#### Tricky Follow-up:

* **Question:** *What happens if you omit `argtypes` and `restype` on a 64-bit C function returning a 64-bit pointer in `ctypes`?*

* **Answer:** `ctypes` defaults to 32-bit `c_int` return type, truncating 64-bit memory addresses and causing segmentation faults.



In [ ]:
import ctypes

class Point3D(ctypes.Structure):
    _fields_ = [
        ("x", ctypes.c_double),
        ("y", ctypes.c_double),
        ("z", ctypes.c_double)
    ]

pt = Point3D(1.5, 2.5, 3.5)
print("1. C-Structure Size via ctypes:", ctypes.sizeof(pt), "bytes (3 doubles x 8 bytes)")
print("2. C-Structure Field Values:   ", pt.x, pt.y, pt.z)


### Q139: What are weakref.ref and weakref.WeakValueDictionary and how do they prevent circular reference memory leaks?

#### Verbal Answer:

Weak references allow referencing objects without incrementing their reference count:

1. **`weakref.ref(obj)`**:
   * Returns a weak reference pointer. If `obj` reaches refcount 0, it is deallocated immediately, and calling `r()` returns `None`.

2. **`weakref.WeakValueDictionary`**:
   * A dictionary holding weak references to values.
   * When an object is deleted elsewhere in the program, its entry is **automatically removed from the dictionary**!
   * Ideal for object pools, flyweight patterns, and memory caches.

---

#### Tricky Follow-up:

* **Question:** *Can you create weak references to built-in types like `list` or `dict`?*

* **Answer:** No. Built-in `list` and `dict` do not support weak references directly unless subclassed.



In [ ]:
import weakref

class CacheItem:
    def __init__(self, name):
        self.name = name

weak_cache = weakref.WeakValueDictionary()
item = CacheItem("Session_123")
weak_cache["session"] = item

print("1. Object exists in weak cache:", "session" in weak_cache)
del item  # Deleting strong reference
print("2. Auto-evicted from weak cache after del:", "session" in weak_cache)


### Q140: What is Python's __future__ module and how do compiler flags like annotations change runtime semantics?

#### Verbal Answer:

The `__future__` module enables future language features in current Python versions:

1. **`from __future__ import annotations` (PEP 563)**:
   * Postpones evaluation of type annotations.
   * Annotations are stored as raw strings in `__annotations__` without executing at class definition time (allows circular self-referencing types `def f() -> Node:` and accelerates module load time).

---

#### Tricky Follow-up:

* **Question:** *Where must `from __future__ import ...` statements be placed in a Python file?*

* **Answer:** They MUST be placed at the very top of the file before any other executable code or imports.



In [ ]:
from __future__ import annotations

class Node:
    def get_parent(self) -> Node | None:  # Self-referencing type hint without quotes!
        return None

print("Annotations stored as strings via __future__.annotations:")
print(Node.get_parent.__annotations__)


## Category 15: High-Performance, Compilation & Production Anti-Patterns


### Q141: How do you compile Python functions to C-speed with @numba.jit(nopython=True) and what are its constraints?

#### Verbal Answer:

Numba JIT (Just-In-Time) LLVM Compilation:

1. **`@numba.njit` / `@numba.jit(nopython=True)`**:
   * Compiles Python math functions into native machine code.
   * Achieves C/Fortran speeds by running in pure C-loops without Python boxing.

2. **Constraints**:
   * Operates strictly on numeric dtypes and NumPy arrays.
   * Cannot use arbitrary Python objects, standard dicts, or strings inside `nopython` mode.

---

#### Tricky Follow-up:

* **Question:** *How does `nogil=True` in `@numba.jit` enable multi-core parallel processing?*

* **Answer:** It releases Python's GIL during the JIT loop, allowing multiple native threads to execute across all CPU cores simultaneously.



In [ ]:
import numpy as np

# Pure Python / NumPy baseline representation
def monte_carlo_pi(n):
    acc = 0
    for i in range(n):
        acc += i * 2
    return acc

print("Numba JIT transforms numerical loops into compiled native LLVM assembly.")
print("Computed result:", monte_carlo_pi(1000))


### Q142: How does Cython bridge C data structures, cdef static types, and nogil blocks to achieve 100x speedups?

#### Verbal Answer:

Cython C-Extension Architecture:

1. **Static C-Types (`cdef`)**:
   * `cdef int i, total = 0` declares raw C types, eliminating Python object boxing.

2. **`with nogil:` Blocks**:
   * Releases the GIL for pure C sections, allowing OpenMP multi-threading:
   ```cython
   from cython.parallel import prange
   for i in prange(n, nogil=True): ...
   ```

---

#### Tricky Follow-up:

* **Question:** *What happens if a Cython `with nogil:` block attempts to access a Python object?*

* **Answer:** The Cython compiler raises a compilation error because accessing Python objects requires acquiring the GIL.



In [ ]:
print("Cython pattern: cdef int / double compiles directly into native C extensions (.pyd / .so).")


### Q143: What is CFFI (C Foreign Function Interface) and why is it preferred over raw CPython C-API extensions?

#### Verbal Answer:

CFFI (C Foreign Function Interface):

1. **Advantages over CPython C-API**:
   * Compatible across both CPython and PyPy JIT.
   * Write plain C header signatures (`ffi.cdef(...)`) directly in Python without writing verbose CPython boilerplate (`PyArg_ParseTuple`, `Py_INCREF`).

---

#### Tricky Follow-up:

* **Question:** *What are the two modes of CFFI?*

* **Answer:** ABI mode (dynamic runtime lookup) and API mode (ahead-of-time C compilation via a C compiler).



In [ ]:
print("CFFI standardizes foreign C function binding with pure C header definitions.")


### Q144: How does PyPy JIT differ from CPython and why can PyPy be slower on C-extension heavy workloads (NumPy/Pandas)?

#### Verbal Answer:

PyPy JIT vs CPython:

1. **PyPy Tracing JIT**:
   * Analyzes running loops and compiles hot loops directly to machine code (4x to 7x faster for pure Python).

2. **The C-Extension Bottleneck**:
   * PyPy must emulate the CPython C-API through an adapter layer (`cpyext`), introducing heavy marshaling overhead when calling NumPy, Pandas, or PyTorch.

---

#### Tricky Follow-up:

* **Question:** *When is PyPy the ideal production choice?*

* **Answer:** For long-running, pure-Python web services, socket servers, and business logic pipelines that do not rely heavily on C-extensions.



In [ ]:
import sys
print("Current Runtime Implementation:", sys.implementation.name)


### Q145: What is the __missing__ dunder method and how do you build custom Self-Populating Caches?

#### Verbal Answer:

`__missing__(self, key)` dunder hook:

1. **Mechanism**:
   * Called by `dict.__getitem__` **ONLY when a key is not found** in a dictionary subclass.
   * Can compute, store, and return the missing value dynamically.

---

#### Tricky Follow-up:

* **Question:** *Does `__missing__` get called when calling `d.get(key)`?*

* **Answer:** No. `d.get(key)` bypasses `__missing__`.



In [ ]:
class FibonacciCache(dict):
    def __missing__(self, n):
        if n < 2:
            val = n
        else:
            val = self[n - 1] + self[n - 2]
        self[n] = val
        return val

fib_cache = FibonacciCache()
print("Computed and Cached Fibonacci(10) via __missing__:", fib_cache[10])


### Q146: What are the performance implications of Global Variable lookups vs Local Variable lookups in CPython?

#### Verbal Answer:

Local vs Global Variable Lookup Performance:

1. **Local Variables (`LOAD_FAST`)**:
   * Stored in a fixed-size C array on the stack frame. Accessed by integer index in O(1) time.

2. **Global Variables (`LOAD_GLOBAL`)**:
   * Looked up via hash table lookups in the global dictionary, then the built-in dictionary.

3. **Optimization Pattern**:
   * Storing global functions (like `math.sin` or `math.sqrt`) as local default arguments (`def loop(val, sin=math.sin):`) speeds up tight loops by 20-30%.

---

#### Tricky Follow-up:

* **Question:** *How can you verify that a function uses `LOAD_FAST` vs `LOAD_GLOBAL`?*

* **Answer:** Disassemble the function bytecode using `dis.dis(func)`.



In [ ]:
import dis

def local_demo(x):
    y = x + 10
    return y

print("Disassembly showing LOAD_FAST for local variables:")
dis.dis(local_demo)


### Q147: How do you implement Structured Graceful Shutdown in Python microservices using signal handlers (SIGTERM, SIGINT)?

#### Verbal Answer:

Graceful Shutdown Architecture:

1. **Signals**:
   * `SIGTERM` (sent by Kubernetes/Docker to stop pods).
   * `SIGINT` (`Ctrl+C`).

2. **Signal Handling Pattern**:
   * Register `signal.signal(signal.SIGTERM, handler)`.
   * Set a `threading.Event()` shutdown flag to allow in-flight database transactions and requests to complete before exiting.

---

#### Tricky Follow-up:

* **Question:** *Can signal handlers be registered on background worker threads in Python?*

* **Answer:** No. Python signal handlers can strictly only be registered on the **Main Thread**.



In [ ]:
import signal

shutdown_requested = False

def sigterm_handler(signum, frame):
    global shutdown_requested
    shutdown_requested = True
    print(f"[SHUTDOWN] Received signal {signum}. Commencing graceful drain...")

# Simulate handler
sigterm_handler(signal.SIGTERM, None)
print("Graceful Shutdown Flag State:", shutdown_requested)


### Q148: What are the security hazards of eval(), exec(), and os.system() and what are their secure alternatives?

#### Verbal Answer:

Code Injection Hazards and Secure Alternatives:

1. **`eval()` & `exec()`**:
   * Executing untrusted user strings allows Arbitrary Remote Code Execution (RCE).
   * **Secure Alternative**: Use `ast.literal_eval()` for safely parsing JSON/Python literals (strings, numbers, dicts, lists).

2. **`os.system(cmd)`**:
   * Vulnerable to Command Injection (shell meta-characters `cmd; rm -rf /`).
   * **Secure Alternative**: Use `subprocess.run(['cmd', 'arg'], shell=False)`.

---

#### Tricky Follow-up:

* **Question:** *Why is `ast.literal_eval()` completely safe against malicious function calls?*

* **Answer:** Because it strictly parses the Abstract Syntax Tree for literal data structures and raises `ValueError` if any operator, function call, or expression is present.



In [ ]:
import ast

user_payload = "{'user_id': 101, 'active': True, 'score': 98.5}"

# Safe parsing via ast.literal_eval
safe_data = ast.literal_eval(user_payload)
print("Safely Parsed Data via ast.literal_eval:", safe_data)


### Q149: How do you profile CPU and Memory bottlenecks using cProfile, pstats, and line_profiler?

#### Verbal Answer:

Profiling Tooling Suite:

1. **`cProfile`**:
   * Standard library deterministic profiler. Measures cumulative time and call counts per function.

2. **`pstats.Stats`**:
   * Formats, filters, and sorts profiling output (`stats.sort_stats('cumtime').print_stats(10)`).

3. **`line_profiler` (`@profile`)**:
   * Measures execution time spent **line-by-line** inside a single function.

---

#### Tricky Follow-up:

* **Question:** *What is the difference between `tottime` and `cumtime` in `cProfile` output?*

* **Answer:** `tottime` is the time spent strictly inside the function body; `cumtime` includes time spent inside sub-functions called by that function.



In [ ]:
import cProfile
import pstats
import io

def target_pipeline():
    return sum(x**2 for x in range(10000))

pr = cProfile.Profile()
pr.enable()
target_pipeline()
pr.disable()

s = io.StringIO()
ps = pstats.Stats(pr, stream=s).sort_stats('cumtime')
ps.print_stats(3)
print("cProfile Output Summary:", s.getvalue()[:200])


### Q150: What are the Top 5 Deadliest Production Anti-Patterns in Python and their Senior Engineering Solutions?

#### Verbal Answer:

The 5 Deadliest Production Anti-Patterns in Python:

1. **Anti-Pattern 1: Mutable Default Arguments (`def f(x=[])`)**:
   * *Trap*: Shared state across all callers.
   * *Solution*: Use `x=None` and initialize inside function.

2. **Anti-Pattern 2: Bare `except:` & Catching `BaseException`**:
   * *Trap*: Silences `KeyboardInterrupt` / `SystemExit` and masks syntax bugs.
   * *Solution*: Catch specific domain exceptions or `except Exception:`.

3. **Anti-Pattern 3: String Concatenation in Loops (`s += chunk`)**:
   * *Trap*: Quadratic O(N^2) memory reallocation.
   * *Solution*: Collect in a list and call `''.join(list)` in O(N) time.

4. **Anti-Pattern 4: Late Binding Closures in Loops**:
   * *Trap*: All lambdas evaluate to the final loop value.
   * *Solution*: Use default argument binding `lambda i=i: i`.

5. **Anti-Pattern 5: Blocking I/O inside Asyncio Event Loops**:
   * *Trap*: Freezes the entire server event loop.
   * *Solution*: Offload to `asyncio.to_thread()` or use asynchronous client libraries (e.g. `httpx`, `asyncpg`).

---

#### Tricky Follow-up:

* **Question:** *What is the single most valuable static analysis configuration for professional Python codebases?*

* **Answer:** Enforcing strict static type checking with `mypy --strict` alongside `ruff` linting in automated CI/CD pipelines.



In [ ]:
# Correct, production-grade string assembly in O(N) time
chunks = [f"part_{i}" for i in range(5)]
assembled_string = "-".join(chunks)

print("Senior Production-Grade String Assembly Result:", assembled_string)
